# LeetCode 218 — The Skyline Problem bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def getSkyline(self, buildings: List[List[int]]) -> List[List[int]]:

```

Mục tiêu của bài học:

1. Hiểu đúng đề: **đường chân trời (skyline)** là gì, **key point** (điểm then chốt) là gì, vì sao phải **nén** các đoạn ngang cùng chiều cao.
2. Biết "nhìn" bài toán qua **hàm chiều cao `h(x)`** và hình vẽ ASCII; nắm chắc quy ước **nửa khoảng** `[left, right)`.
3. Đi từng bước: tô từng ô (ngây thơ) → brute force trên **tọa độ quan trọng** `O(n²)` → **sweep line + max-heap** `O(n log n)` (cách chính), hiểu rõ "cách cũ vướng ở đâu → mấu chốt cải tiến".
4. Học từ đầu **heap / `heapq`**, mẹo **max-heap bằng số âm** và kỹ thuật **xóa lười (lazy deletion)**.
5. Hiểu bản **sự kiện (event)** với **quy tắc sắp xếp khi trùng x** (vào trước ra, cao trước thấp), bản **chia để trị** (merge hai skyline như merge sort) và bản **multiset tự cài** bằng `bisect`.
6. Tự viết lại lời giải, tránh các lỗi kinh điển và nhận dạng pattern "đường quét + tập đang hoạt động".

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> **Đường chân trời** (skyline) của một thành phố là đường viền ngoài của hình bóng (silhouette) do tất cả các tòa nhà trong thành phố tạo thành khi nhìn từ xa. Cho vị trí và chiều cao của mọi tòa nhà, hãy trả về **skyline** mà các tòa nhà này cùng tạo thành.
>
> Thông tin hình học của mỗi tòa nhà nằm trong mảng `buildings`, với `buildings[i] = [left_i, right_i, height_i]`:
>
> - `left_i` là tọa độ x của **mép trái** tòa nhà thứ `i`.
> - `right_i` là tọa độ x của **mép phải** tòa nhà thứ `i`.
> - `height_i` là **chiều cao** của tòa nhà thứ `i`.
>
> Có thể giả sử mọi tòa nhà là các **hình chữ nhật hoàn hảo** đặt trên một mặt đất tuyệt đối phẳng ở độ cao `0`.
>
> Skyline được biểu diễn bằng một danh sách các **"key point"** sắp xếp theo tọa độ x, dạng `[[x1,y1],[x2,y2],...]`. Mỗi key point là **đầu mút trái** của một đoạn nằm ngang nào đó trong skyline, **trừ điểm cuối cùng** của danh sách — điểm này luôn có tọa độ y bằng `0` và dùng để đánh dấu nơi skyline kết thúc, tức chỗ tòa nhà ngoài cùng bên phải kết thúc. Mọi phần mặt đất nằm giữa tòa nhà trái nhất và tòa nhà phải nhất đều là một phần của đường viền skyline.
>
> **Lưu ý:** Trong skyline đầu ra **không được có các đoạn nằm ngang liên tiếp cùng chiều cao**. Chẳng hạn, `[...,[2 3],[4 5],[7 5],[11 5],[12 7],...]` là không chấp nhận được; ba đoạn cao 5 phải được gộp thành một trong đầu ra cuối cùng như sau: `[...,[2 3],[4 5],[12 7],...]`.

**Example 1**

```text
Input: buildings = [[2,9,10],[3,7,15],[5,12,12],[15,20,10],[19,24,8]]
Output: [[2,10],[3,15],[7,12],[12,0],[15,10],[20,8],[24,0]]
Explanation:
Hình A (trên LeetCode) vẽ các tòa nhà của input.
Hình B vẽ skyline do các tòa nhà đó tạo thành. Các chấm đỏ trong hình B
là các key point của danh sách output.
```

(Ta sẽ tự vẽ lại Hình A và Hình B bằng ASCII ở mục 2.3.)

**Example 2**

```text
Input: buildings = [[0,2,3],[2,5,3]]
Output: [[0,3],[5,0]]
```

**Constraints**

- `1 <= buildings.length <= 10^4`
- `0 <= left_i < right_i <= 2^31 - 1`
- `1 <= height_i <= 2^31 - 1`
- `buildings` được sắp xếp theo `left_i` theo thứ tự **không giảm** (non-decreasing).

### Đọc đề thật chậm — 5 chi tiết quyết định

| Chi tiết trong đề | Ý nghĩa thực tế khi code |
|---|---|
| Key point là **đầu mút trái** của một đoạn ngang | Mỗi khi chiều cao skyline **thay đổi** tại `x`, ghi `[x, chiều cao mới]`. |
| Điểm cuối luôn có `y = 0` | Sau tòa nhà phải nhất, skyline "rơi" xuống đất → luôn kết thúc bằng `[max right, 0]`. |
| Mặt đất giữa các tòa thuộc skyline | Example 1: từ x=12 đến x=15 không có tòa nào → phải có điểm `[12, 0]`. |
| Không có 2 đoạn liên tiếp cùng chiều cao | Example 2: hai tòa cao 3 chạm nhau tại x=2 → **không** được ghi `[2, 3]`. |
| `right_i` là **mép phải** | Tại đúng `x = right_i`, tòa nhà **đã kết thúc** → dùng nửa khoảng `[left, right)` (mục 2.1). |

Ký hiệu dùng suốt notebook:

- `n` = số tòa nhà, `n = len(buildings)`, `1 <= n <= 10^4`.
- Mỗi tòa viết gọn là `[l, r, h]` (left, right, height).
- `h(x)` = chiều cao của skyline tại tọa độ `x`.

Để ý thêm: tọa độ có thể lớn tới `2^31 - 1 ≈ 2,1 tỉ`. Chi tiết nhỏ này sẽ "giết" cách ngây thơ nhất ở mục 3. Còn chi tiết "`buildings` đã sắp theo `left`" sẽ giúp cách chính gọn hơn ở mục 5.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

ex1 = [[2, 9, 10], [3, 7, 15], [5, 12, 12], [15, 20, 10], [19, 24, 8]]
out1 = [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]

ex2 = [[0, 2, 3], [2, 5, 3]]
out2 = [[0, 3], [5, 0]]

# Đặt tên A, B, C, D, E cho 5 tòa của Example 1 để dễ nói chuyện
for name, (l, r, h) in zip("ABCDE", ex1):
    print(f"Tòa {name}: left = {l:>2}, right = {r:>2}, height = {h:>2}  -> rộng {r - l}")
print("n =", len(ex1))
print("Output mong đợi:", out1)

Tòa A: left =  2, right =  9, height = 10  -> rộng 7
Tòa B: left =  3, right =  7, height = 15  -> rộng 4
Tòa C: left =  5, right = 12, height = 12  -> rộng 7
Tòa D: left = 15, right = 20, height = 10  -> rộng 5
Tòa E: left = 19, right = 24, height =  8  -> rộng 5
n = 5
Output mong đợi: [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]


## 2. Nền tảng: hình chữ nhật, nửa khoảng, hàm chiều cao, key point

### 2.1 Một tòa nhà = một hình chữ nhật; quy ước nửa khoảng `[l, r)`

Tòa A = `[2, 9, 10]` chiếm trục x từ 2 đến 9, cao 10. Câu hỏi quan trọng: **tại đúng x = 9**, tòa này còn "đứng" không?

Theo đề, `right = 9` là **mép phải**: tại x = 9 bức tường rơi thẳng xuống. Đoạn ngang trên nóc A bắt đầu từ x = 2 và **kết thúc** tại x = 9; từ x = 9 trở đi, đường viền đi theo đoạn ngang khác. Vì vậy ta quy ước tòa nhà **phủ** điểm x khi:

```python
l <= x < r     # nửa khoảng [l, r): lấy mép trái, KHÔNG lấy mép phải
```

Vì sao không dùng `l <= x <= r`? Nhìn Example 1 tại x = 7: tòa B = `[3, 7, 15]` vừa kết thúc. Nếu vẫn coi B phủ x = 7 thì chiều cao tại 7 là 15 thay vì 12, và key point `[7, 12]` biến mất. Ta sẽ chạy thử đúng lỗi này ở mục 13.

| x | `2 <= x < 9`? | Tòa A có phủ x? |
|---:|:---:|---|
| 1 | False | chưa tới |
| 2 | True | mép trái: **có** |
| 8 | True | có |
| 9 | False | mép phải: **không** (đã hết) |
| 10 | False | đã qua |

Trong Python, `l <= x < r` là phép so sánh **nối chuỗi** (chained comparison): nó có nghĩa `l <= x and x < r`.

In [2]:
def covers(building, x):
    # Tòa building = [l, r, h] có phủ tọa độ x không? (nửa khoảng [l, r))
    l, r, h = building
    return l <= x < r


A = [2, 9, 10]
for x in [1, 2, 8, 9, 10]:
    print(f"x = {x:>2}: covers(A, x) = {covers(A, x)}")

x =  1: covers(A, x) = False
x =  2: covers(A, x) = True
x =  8: covers(A, x) = True
x =  9: covers(A, x) = False
x = 10: covers(A, x) = False


### 2.2 Hàm chiều cao `h(x)`

Đứng tại vị trí `x` và nhìn thẳng lên trời: thứ chặn tầm mắt là **tòa cao nhất đang phủ `x`**. Nếu không có tòa nào thì nhìn thấy mặt đất, chiều cao 0.

```text
h(x) = max( height của mọi tòa có l <= x < r ),   hoặc 0 nếu không có tòa nào
```

Tính tay với Example 1 (A=[2,9,10], B=[3,7,15], C=[5,12,12], D=[15,20,10], E=[19,24,8]):

| x | Tòa đang phủ x | h(x) |
|---:|---|---:|
| 1 | (không có) | 0 |
| 4 | A(10), B(15) | 15 |
| 7 | A(10), C(12) — B đã hết vì `7 < 7` là False | 12 |
| 10 | C(12) — A đã hết tại 9 | 12 |
| 13 | (không có) | 0 |
| 19 | D(10), E(8) | 10 |

**Skyline chính là "đồ thị" của hàm `h(x)`.** Cả bài toán quy về: mô tả gọn đồ thị này bằng danh sách các điểm mà tại đó `h(x)` **thay đổi**.

In [3]:
def height_at(buildings, x):
    # Chiều cao skyline tại x = tòa cao nhất đang phủ x (không có tòa nào thì 0)
    best = 0
    for b in buildings:
        if covers(b, x):
            best = max(best, b[2])
    return best


for x in [1, 4, 7, 10, 13, 19]:
    print(f"h({x:>2}) = {height_at(ex1, x)}")

h( 1) = 0
h( 4) = 15
h( 7) = 12
h(10) = 12
h(13) = 0
h(19) = 10


### 2.3 Vẽ thành phố bằng ASCII

Nhìn thấy hình giúp hiểu đề nhanh gấp nhiều lần. Ta vẽ trên lưới ô vuông 1×1:

- Mỗi **cột** ký tự ứng với đoạn đơn vị `[x, x+1)` trên trục x. Đọc số x ở 2 dòng thước dưới cùng: dòng trên là **hàng chục**, dòng dưới là **hàng đơn vị** (cột có `1` ở dòng trên và `5` ở dòng dưới là x = 15).
- Mỗi **hàng** ứng với độ cao `y` (số ghi bên trái).

**Hình A — các tòa nhà.** Ô ở cột x, hàng y được tô bằng chữ cái của một tòa phủ x và cao ít nhất y. Nếu nhiều tòa cùng thỏa, ta in tòa **thấp nhất** — tưởng tượng tòa thấp đứng phía trước, tòa cao đứng phía sau — nhờ vậy **nóc của mọi tòa đều lộ ra**, giống hình A trên LeetCode.

**Hình B — skyline.** Ô được tô `#` nếu `y <= h(x)` (nằm dưới đường viền). Mỗi key point `[x, y]` được đánh dấu `*` ở **ô góc trên-trái** của đoạn ngang bắt đầu từ nó; key point có `y = 0` được đánh dấu `*` ngay trên dòng mặt đất.

Hai hàm vẽ chỉ dùng cho tọa độ nhỏ (vài chục) để học — không liên quan tới code nộp bài.

In [4]:
def draw_buildings(buildings):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    H = max(h for _, _, h in buildings)          # chiều cao lớn nhất
    W = max(r for _, r, _ in buildings)          # mép phải lớn nhất
    for y in range(H, 0, -1):                    # vẽ từ trên xuống
        row = ""
        for x in range(W + 1):
            best = None                          # tòa thấp nhất phủ ô (x, y)
            for k, (l, r, h) in enumerate(buildings):
                if l <= x < r and h >= y and (best is None or h < buildings[best][2]):
                    best = k
            row += letters[best] if best is not None else " "
        print(f"{y:>3} |" + row)
    print("    +" + "-" * (W + 1))
    print("     " + "".join(str(x // 10) if x % 10 == 0 else " " for x in range(W + 1)))
    print("     " + "".join(str(x % 10) for x in range(W + 1)))


print("Hình A — Example 1")
draw_buildings(ex1)

Hình A — Example 1
 15 |   BBBB                  
 14 |   BBBB                  
 13 |   BBBB                  
 12 |   BBCCCCCCC             
 11 |   BBCCCCCCC             
 10 |  AAAAAAACCC   DDDDD     
  9 |  AAAAAAACCC   DDDDD     
  8 |  AAAAAAACCC   DDDDEEEEE 
  7 |  AAAAAAACCC   DDDDEEEEE 
  6 |  AAAAAAACCC   DDDDEEEEE 
  5 |  AAAAAAACCC   DDDDEEEEE 
  4 |  AAAAAAACCC   DDDDEEEEE 
  3 |  AAAAAAACCC   DDDDEEEEE 
  2 |  AAAAAAACCC   DDDDEEEEE 
  1 |  AAAAAAACCC   DDDDEEEEE 
    +-------------------------
     0         1         2    
     0123456789012345678901234


In [5]:
def draw_skyline(buildings, key_points):
    H = max(h for _, _, h in buildings)
    W = max(r for _, r, _ in buildings)
    marks = {(x, y) for x, y in key_points}      # set các key point để tra nhanh
    for y in range(H, 0, -1):
        row = ""
        for x in range(W + 1):
            if (x, y) in marks:
                row += "*"
            elif height_at(buildings, x) >= y:
                row += "#"
            else:
                row += " "
        print(f"{y:>3} |" + row)
    print("    +" + "".join("*" if (x, 0) in marks else "-" for x in range(W + 1)))
    print("     " + "".join(str(x // 10) if x % 10 == 0 else " " for x in range(W + 1)))
    print("     " + "".join(str(x % 10) for x in range(W + 1)))


print("Hình B — Example 1, key point:", out1)
draw_skyline(ex1, out1)
print()
print("Example 2, key point:", out2)
draw_skyline(ex2, out2)

Hình B — Example 1, key point: [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
 15 |   *###                  
 14 |   ####                  
 13 |   ####                  
 12 |   ####*####             
 11 |   #########             
 10 |  *#########   *####     
  9 |  ##########   #####     
  8 |  ##########   #####*### 
  7 |  ##########   ######### 
  6 |  ##########   ######### 
  5 |  ##########   ######### 
  4 |  ##########   ######### 
  3 |  ##########   ######### 
  2 |  ##########   ######### 
  1 |  ##########   ######### 
    +------------*-----------*
     0         1         2    
     0123456789012345678901234

Example 2, key point: [[0, 3], [5, 0]]
  3 |*#### 
  2 |##### 
  1 |##### 
    +-----*
     0     
     012345


Đọc hình B của Example 1 từ trái sang phải, đối chiếu với Hình A:

| Tại x | Chuyện gì xảy ra? | Có key point? |
|---:|---|---|
| 2 | A bắt đầu, skyline nhảy từ 0 lên 10 | `[2, 10]` |
| 3 | B (15) bắt đầu, cao hơn A | `[3, 15]` |
| 5 | C (12) bắt đầu nhưng **thấp hơn B** → bị che | **không** |
| 7 | B kết thúc; tòa cao nhất còn lại là **C (12)**, không phải A (10) | `[7, 12]` |
| 9 | A kết thúc nhưng đang bị C che → không ai thấy | **không** |
| 12 | C kết thúc, không còn tòa nào → rơi xuống đất | `[12, 0]` |
| 15 | D (10) bắt đầu | `[15, 10]` |
| 19 | E (8) bắt đầu nhưng thấp hơn D | **không** |
| 20 | D kết thúc, còn E | `[20, 8]` |
| 24 | E kết thúc — điểm kết thúc skyline | `[24, 0]` |

Example 2: hai tòa cao 3 chạm nhau tại x = 2 tạo thành **một** đoạn ngang liền từ 0 tới 5, nên chỉ có `[0, 3]` và `[5, 0]`.

Hai bài học rút ra:

1. Key point **chỉ** có thể xuất hiện tại mép trái/phải của các tòa.
2. Nhưng **không phải mép nào cũng sinh key point** — chỉ những mép làm `h(x)` thay đổi (x = 5, 9, 19 không sinh điểm nào).

### 2.4 "Nén" các đoạn ngang cùng chiều cao

Giả sử ta đã có danh sách cặp `(x, h(x))` theo x tăng dần. Lấy luôn ví dụ trong phần Lưu ý của đề (thêm một điểm kết thúc cho đủ):

```text
(2,3) (4,5) (7,5) (11,5) (12,7) (14,0)
```

Ba điểm cao 5 liên tiếp tạo thành **một** đoạn ngang dài từ 4 đến 12, nên chỉ giữ điểm đầu tiên `[4, 5]`.

Quy tắc nén: giữ biến `prev` = chiều cao vừa ghi gần nhất, ban đầu `prev = 0` (bên trái mọi tòa là mặt đất). Với mỗi `(x, h)`:

- nếu `h != prev` → ghi `[x, h]` và cập nhật `prev = h` (đoạn ngang mới bắt đầu);
- nếu `h == prev` → bỏ qua (đoạn ngang cũ vẫn đang tiếp diễn).

Mọi cách giải trong notebook đều kết thúc bằng đúng quy tắc 3 dòng này.

In [6]:
def compress_heights(samples):
    # samples: list các cặp (x, h) theo x tăng dần -> list key point đã nén
    res = []
    prev = 0
    for x, h in samples:
        if h != prev:
            res.append([x, h])
            prev = h
    return res


samples = [(2, 3), (4, 5), (7, 5), (11, 5), (12, 7), (14, 0)]
print(compress_heights(samples))

[[2, 3], [4, 5], [12, 7], [14, 0]]


## 3. Cách 1 — Brute force: từ "tô từng ô" đến "chỉ xét tọa độ quan trọng"

### 3.1 Bước 1a — Tô từng ô (cách ngây thơ nhất)

Cách người mới hay nghĩ ra đầu tiên chính là cách ta vừa vẽ hình: đi qua **từng số nguyên** `x` từ `min left` tới `max right`, tính `h(x)` bằng `height_at`, rồi nén bằng quy tắc `prev`.

```text
prev = 0
for x = min_left, min_left + 1, ..., max_right:
    cur = h(x)                     # duyệt cả n tòa
    nếu cur != prev: ghi [x, cur], prev = cur
```

Vì sao kiểm tra các số nguyên là **đủ**? Mọi `left`, `right` đều là số nguyên, nên bên trong một đoạn `[x, x+1)` không có tòa nào bắt đầu hay kết thúc → `h` không đổi trên cả đoạn. Chiều cao chỉ có thể đổi **tại** các số nguyên.

Cú pháp mới trong cell dưới: `min(l for l, _, _ in buildings)` duyệt từng tòa, tách 3 phần tử, lấy `l` rồi tìm nhỏ nhất. Dấu `_` là tên biến quy ước cho giá trị "không dùng tới".

In [7]:
def skyline_grid(buildings):
    lo = min(l for l, _, _ in buildings)      # mép trái nhỏ nhất
    hi = max(r for _, r, _ in buildings)      # mép phải lớn nhất
    res, prev = [], 0
    for x in range(lo, hi + 1):               # TỪNG số nguyên x
        cur = height_at(buildings, x)         # duyệt n tòa
        if cur != prev:                       # chiều cao đổi -> key point
            res.append([x, cur])
            prev = cur
    return res


print(skyline_grid(ex1))
print(skyline_grid(ex2))
assert skyline_grid(ex1) == out1
assert skyline_grid(ex2) == out2
print("Bước 1a đúng với cả 2 ví dụ.")

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
[[0, 3], [5, 0]]
Bước 1a đúng với cả 2 ví dụ.


### 3.2 Bước 1a vướng ở đâu? → Quan sát then chốt

Gọi `W = max right - min left` là độ rộng thành phố. Ta tính `h(x)` cho `W + 1` giá trị x, mỗi lần duyệt `n` tòa → **`O(n · W)`**.

Example 1 có W = 22 nên chạy tức thì. Nhưng constraints cho phép một tòa `[0, 2^31 - 1, ...]`: W ≈ 2,1 **tỉ**. Python làm cỡ vài chục triệu phép đơn giản mỗi giây, nên chỉ riêng vòng lặp qua 2,1 tỉ giá trị x đã mất nhiều phút — chưa kể mỗi x còn duyệt `n` tòa. Không thể nộp.

**Quan sát cứu cánh** (nhìn lại Hình B): đồ thị `h(x)` là một **hàm bậc thang**. Nó chỉ có thể đổi giá trị tại **mép** của một tòa (nơi một tòa bắt đầu hoặc kết thúc). Giữa hai mép liên tiếp, không tòa nào xuất hiện hay biến mất → tập tòa đang phủ x giữ nguyên → `h(x)` giữ nguyên.

→ Chỉ cần tính `h(x)` tại các **tọa độ quan trọng** (critical points) = tập mọi `left` và `right`. Có **tối đa `2n`** điểm, bất kể tọa độ lớn cỡ nào.

Cú pháp mới trong cell dưới:

- `{x for l, r, _ in ex1 for x in (l, r)}` là **set comprehension** hai tầng: với mỗi tòa, lấy cả `l` lẫn `r`; `set` tự bỏ phần tử trùng (Example 2 có x = 2 xuất hiện hai lần). Bọc ngoài bằng `sorted(...)` để được list tăng dần.
- `zip(critical, critical[1:])` ghép từng cặp **liên tiếp**: `(2,3), (3,5), (5,7), ...` — mỗi cặp `(a, b)` là một đoạn `[a, b)` giữa hai tọa độ quan trọng.

In [8]:
critical = sorted({x for l, r, _ in ex1 for x in (l, r)})
print("Tọa độ quan trọng:", critical)
print("Số tọa độ quan trọng:", len(critical), "<= 2n =", 2 * len(ex1))

# Kiểm chứng: giữa hai tọa độ quan trọng liên tiếp, h(x) chỉ có MỘT giá trị
for a, b in zip(critical, critical[1:]):
    values = {height_at(ex1, x) for x in range(a, b)}
    print(f"đoạn [{a:>2}, {b:>2}): tập giá trị h(x) = {values}")

Tọa độ quan trọng: [2, 3, 5, 7, 9, 12, 15, 19, 20, 24]
Số tọa độ quan trọng: 10 <= 2n = 10
đoạn [ 2,  3): tập giá trị h(x) = {10}
đoạn [ 3,  5): tập giá trị h(x) = {15}
đoạn [ 5,  7): tập giá trị h(x) = {15}
đoạn [ 7,  9): tập giá trị h(x) = {12}
đoạn [ 9, 12): tập giá trị h(x) = {12}
đoạn [12, 15): tập giá trị h(x) = {0}
đoạn [15, 19): tập giá trị h(x) = {10}
đoạn [19, 20): tập giá trị h(x) = {10}
đoạn [20, 24): tập giá trị h(x) = {8}


### 3.3 Bước 1b — Brute force trên tọa độ quan trọng

Thuật toán:

1. `xs` = các tọa độ quan trọng, sắp tăng dần.
2. Với mỗi `x` trong `xs`: tính `cur = h(x)` bằng cách duyệt **mọi** tòa.
3. Nén: chỉ ghi `[x, cur]` khi `cur != prev`.

Vì sao điểm cuối luôn là `[max_right, 0]`? Tại `x = max_right`, mọi tòa đều có `r <= max_right`, nên không tòa nào phủ x → `h = 0`. Còn ở điểm dừng ngay trước đó, tòa có `r = max_right` vẫn đang phủ (vì `l` của nó cũng là một điểm dừng và nhỏ hơn `max_right`) → chiều cao > 0. Vậy chắc chắn có thay đổi → ghi `[max_right, 0]`. Lập luận tương tự cho các khoảng mặt đất ở giữa (như `[12, 0]`).

Cách này sẽ là **"trọng tài"** (reference) để kiểm tra chéo các cách nhanh hơn: chậm nhưng gần như không thể sai.

In [9]:
def skyline_bruteforce(buildings):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})   # <= 2n điểm dừng
    res, prev = [], 0
    for x in xs:
        cur = 0
        for l, r, h in buildings:          # duyệt LẠI cả n tòa ở mỗi điểm dừng
            if l <= x < r:
                cur = max(cur, h)
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


print(skyline_bruteforce(ex1))
print(skyline_bruteforce(ex2))
assert skyline_bruteforce(ex1) == out1
assert skyline_bruteforce(ex2) == out2
# Tọa độ khổng lồ không còn là vấn đề:
print(skyline_bruteforce([[0, 2**31 - 1, 5], [10, 2**31 - 2, 7]]))

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
[[0, 3], [5, 0]]
[[0, 5], [10, 7], [2147483646, 5], [2147483647, 0]]


### 3.4 Độ phức tạp và nút thắt của Bước 1b

- Có tối đa `2n` điểm dừng, mỗi điểm duyệt `n` tòa → khoảng `2n²` lần kiểm tra "tòa có phủ x?" → **Time `O(n²)`**.
- **Space `O(n)`** cho `xs` và kết quả.

Với `n = 10^4`: `2·10^8` lần kiểm tra — Python cần cỡ vài chục giây tới cả phút → **TLE** (Time Limit Exceeded). Cell dưới đếm chính xác con số này.

**Nút thắt**: tại mỗi điểm dừng ta tính lại `h(x)` **từ đầu**, duyệt cả những tòa đã kết thúc từ lâu và những tòa chưa bắt đầu. Trong khi đó, khi đường đi từ điểm dừng này sang điểm dừng kế tiếp, **chỉ vài tòa đổi trạng thái** (bắt đầu hoặc kết thúc). Phần lớn công việc là lặp lại vô ích.

Câu hỏi dẫn tới cách chính:

> Có cấu trúc dữ liệu nào giữ **"tập các tòa đang đứng"**, cho phép **thêm** tòa, **bỏ** tòa, và hỏi **"tòa cao nhất là bao nhiêu?"** thật nhanh không?

Câu trả lời: **heap** (hàng đợi ưu tiên).

In [10]:
def count_checks_bruteforce(buildings):
    xs = {x for l, r, _ in buildings for x in (l, r)}
    return len(xs) * len(buildings)          # số lần chạy dòng "if l <= x < r"


for n in [10, 100, 1000, 10000]:
    city = [[10 * i, 10 * i + 5, 1] for i in range(n)]   # n tòa rời nhau -> 2n điểm dừng
    print(f"n = {n:>5}: số lần kiểm tra = {count_checks_bruteforce(city):>12,}")

n =    10: số lần kiểm tra =          200
n =   100: số lần kiểm tra =       20,000
n =  1000: số lần kiểm tra =    2,000,000
n = 10000: số lần kiểm tra =  200,000,000


## 4. Nền tảng: Heap và `heapq` từ con số 0

### 4.1 Min-heap là gì?

**Heap** ("đống") là cấu trúc dữ liệu luôn đưa ra **phần tử nhỏ nhất** ngay lập tức. **Min-heap** là một cây nhị phân thỏa: **cha ≤ các con** → phần tử nhỏ nhất nằm ở gốc. Python lưu heap trong một `list` thường theo quy ước vị trí: gốc ở `heap[0]`, hai con của `heap[i]` là `heap[2*i + 1]` và `heap[2*i + 2]`.

Module `heapq` (thư viện chuẩn) cung cấp:

| Thao tác | Code | Time (heap có `m` phần tử) |
|---|---|---|
| Thêm | `heapq.heappush(h, x)` | `O(log m)` |
| Lấy ra **và xóa** phần tử nhỏ nhất | `heapq.heappop(h)` | `O(log m)` |
| Xem nhỏ nhất (không xóa) | `h[0]` | `O(1)` |
| Biến list bất kỳ thành heap | `heapq.heapify(h)` | `O(m)` |

Hai điều cần nhớ:

- List bên trong heap **không** được sắp xếp hoàn toàn; chỉ bảo đảm `h[0]` nhỏ nhất.
- Heap **không** có thao tác xóa nhanh một phần tử **bất kỳ ở giữa**. Chi tiết này cực kỳ quan trọng với bài này (mục 4.3).

In [11]:
import heapq

h = []
for x in [10, 15, 12, 8]:
    heapq.heappush(h, x)
    print(f"push {x:>2} -> list bên trong = {h}, nhỏ nhất h[0] = {h[0]}")

print("pop ->", heapq.heappop(h), "| còn lại:", h)
print("pop ->", heapq.heappop(h), "| còn lại:", h)

push 10 -> list bên trong = [10], nhỏ nhất h[0] = 10
push 15 -> list bên trong = [10, 15], nhỏ nhất h[0] = 10
push 12 -> list bên trong = [10, 15, 12], nhỏ nhất h[0] = 10
push  8 -> list bên trong = [8, 10, 12, 15], nhỏ nhất h[0] = 8
pop -> 8 | còn lại: [10, 15, 12]
pop -> 10 | còn lại: [12, 15]


### 4.2 Max-heap bằng số âm, và heap chứa tuple

Bài này cần **tòa cao nhất** — tức phần tử **lớn nhất**, trong khi `heapq` chỉ có min-heap. Mẹo kinh điển: **đổi dấu**. Lưu `-h` thay vì `h`: số lớn nhất trở thành số âm **nhỏ nhất** → nằm ở `heap[0]`. Khi đọc ra nhớ đổi dấu lại: `-heap[0]`.

Ví dụ: chiều cao 10, 15, 12 → lưu -10, -15, -12 → nhỏ nhất là -15 → cao nhất là 15.

Heap còn chứa được **tuple**. Python so sánh tuple kiểu "từ điển": so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai, v.v.

```python
(-15, 7) < (-12, 12)   # True  vì -15 < -12, không cần xem phần sau
(-10, 5) < (-10, 9)    # True  vì phần đầu bằng nhau, 5 < 9
```

Trong bài này, mỗi tòa vào heap dưới dạng **`(-height, right)`**:

- phần đầu `-height` quyết định "ai cao nhất" (ai nằm ở đỉnh);
- phần sau `right` là **hạn dùng** của tòa (tọa độ nó kết thúc) — ta cần nó để biết tòa trên đỉnh **còn đứng hay đã hết**.

In [12]:
max_heap = []
for hh in [10, 15, 12]:
    heapq.heappush(max_heap, -hh)            # lưu số âm
print("list bên trong:", max_heap, "-> cao nhất =", -max_heap[0])

print((-15, 7) < (-12, 12), (-10, 5) < (-10, 9))

towers = []
for l, r, hh in ex1[:3]:                    # 3 tòa A, B, C
    heapq.heappush(towers, (-hh, r))
print("đỉnh heap:", towers[0], "-> tòa cao nhất cao", -towers[0][0], "và hết hạn tại x =", towers[0][1])

list bên trong: [-15, -10, -12] -> cao nhất = 15
True True
đỉnh heap: (-15, 7) -> tòa cao nhất cao 15 và hết hạn tại x = 7


### 4.3 Xóa lười (lazy deletion)

**Vấn đề**: khi một tòa kết thúc, ta muốn bỏ nó khỏi heap. Nhưng tòa đó có thể nằm **ở giữa** heap; tìm nó tốn `O(m)`, xóa xong còn phải sắp lại heap.

**Ý tưởng xóa lười**: *đừng vội xóa*. Một tòa đã hết hạn nằm sâu trong heap **chẳng ảnh hưởng gì**, vì ta chỉ bao giờ đọc `heap[0]`. Nó chỉ gây hại khi nó **lên tới đỉnh**. Vậy mỗi lần cần đọc đỉnh tại vị trí x, ta "dọn" đỉnh trước:

```python
while heap and heap[0][1] <= x:   # đỉnh đã hết hạn tại x?
    heapq.heappop(heap)           # vứt nó; đỉnh mới lại được kiểm tra tiếp
```

So sánh đời thường: một chồng phiếu ưu tiên, phiếu nào hết hạn cứ để yên đó; chỉ khi rút phiếu trên cùng mà thấy hết hạn mới vứt đi và xem phiếu tiếp theo.

- Vì sao dùng `<=`? Vì nửa khoảng `[l, r)`: tại đúng `x = r` tòa đã hết.
- Vì sao rẻ? Mỗi tòa vào heap **một lần** và bị pop **nhiều nhất một lần**. Một vòng `while` có thể pop nhiều phần tử, nhưng **tổng** số lần pop trong cả thuật toán không quá `n` → tổng chi phí dọn dẹp chỉ `O(n log n)`. (Lập luận kiểu này gọi là **khấu hao** — amortized.)

Cell dưới cho heap chứa A, B, C rồi lần lượt dọn đỉnh tại x = 7, 9, 12. Ta in `sorted(heap)` chỉ để đọc cho dễ (phần tử đầu là đỉnh).

In [13]:
def clean_top(heap, x):
    removed = []
    while heap and heap[0][1] <= x:
        removed.append(heapq.heappop(heap))
    return removed


heap = []
for l, r, hh in ex1[:3]:                    # A=(-10, 9), B=(-15, 7), C=(-12, 12)
    heapq.heappush(heap, (-hh, r))
print("Ban đầu (A, B, C):", sorted(heap))

for x in [7, 9, 12]:
    removed = clean_top(heap, x)
    top = -heap[0][0] if heap else 0
    print(f"x = {x:>2}: vứt {removed}, heap còn {sorted(heap)}, cao nhất = {top}")

Ban đầu (A, B, C): [(-15, 7), (-12, 12), (-10, 9)]
x =  7: vứt [(-15, 7)], heap còn [(-12, 12), (-10, 9)], cao nhất = 12
x =  9: vứt [], heap còn [(-12, 12), (-10, 9)], cao nhất = 12
x = 12: vứt [(-12, 12), (-10, 9)], heap còn [], cao nhất = 0


## 5. Cách 2 (cách chính) — Sweep line + max-heap + xóa lười

### 5.1 Ý tưởng "đường quét" (sweep line)

Hãy tưởng tượng một **đường thẳng đứng** quét từ trái sang phải qua thành phố. Đường quét chỉ cần **dừng lại** ở các tọa độ quan trọng (mục 3.2). Tại mỗi điểm dừng `x`, làm đúng 4 bước:

1. **Cho vào**: mọi tòa có `left == x` (bắt đầu tại x) được push vào heap dưới dạng `(-h, r)`.
2. **Dọn đỉnh**: pop đỉnh heap chừng nào đỉnh đã hết hạn (`r <= x`).
3. **Đọc chiều cao**: `cur = -heap[0][0]` nếu heap còn phần tử, ngược lại `cur = 0` (mặt đất).
4. **Nén**: nếu `cur != prev` → ghi key point `[x, cur]`, cập nhật `prev`.

So với Cách 1: thay vì duyệt lại cả `n` tòa ở mỗi điểm dừng, ta **giữ lại** tập "các tòa đang đứng" trong heap và chỉ cập nhật phần thay đổi.

Cell 4.3 ở trên chính là bước 2–3 cho A, B, C. Để ý x = 9: tòa A đã hết hạn nhưng **vẫn nằm trong heap** — không sao, vì C (12) đang ở đỉnh "che" nó, chiều cao đọc ra vẫn đúng là 12. Đến x = 12, C bị vứt thì A lộ ra đỉnh và bị vứt luôn trong cùng một vòng `while`.

### 5.2 Ba câu hỏi người mới hay thắc mắc

**(a) Làm sao biết tòa nào có `left == x` mà không duyệt hết?**
Đề bảo đảm `buildings` đã sắp theo `left` tăng dần. Ta giữ một con trỏ `i` (tòa tiếp theo chưa được cho vào). Tại mỗi điểm dừng: `while i < n and buildings[i][0] <= x` thì push `buildings[i]` và tăng `i`. Vì mọi `left` đều là điểm dừng, mỗi tòa được push **đúng một lần**, đúng lúc đường quét tới mép trái của nó. (Đây là lý do đọc constraints rất đáng giá. Nếu đề không cho sắp sẵn, ta tự `sorted` trước — tốn thêm `O(n log n)`, không đổi độ phức tạp.)

**(b) Vì sao phải xử lý *mọi* thay đổi tại cùng một x rồi mới so với `prev`?**
Nhìn Example 2 tại x = 2: tòa `[0,2,3]` kết thúc **và** tòa `[2,5,3]` bắt đầu. Nếu ghi key point giữa chừng (sau khi bỏ tòa cũ, trước khi thêm tòa mới) ta sẽ ghi nhầm `[2, 0]` rồi `[2, 3]`. Gom hết thay đổi tại x rồi mới đọc chiều cao → mỗi x sinh **nhiều nhất một** key point, và đó đúng là chiều cao thật `h(x)`.

**(c) Đỉnh heap có chắc là tòa cao nhất *đang đứng* không, khi trong heap còn "rác"?**
Có. Sau bước dọn, đỉnh là một tòa **còn hiệu lực**. Mọi tòa đang đứng đều nằm trong heap (được push lúc `left`, chưa bị pop vì chưa hết hạn). Đỉnh heap lớn nhất trong **mọi** phần tử của heap → lớn nhất trong các tòa đang đứng. Những tòa hết hạn còn nằm phía dưới chỉ là "rác" vô hại, sớm muộn sẽ bị vứt khi lên đỉnh.

In [14]:
import heapq


def skyline_heap(buildings):
    # Đề bảo đảm buildings đã sắp theo left (không giảm)
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})   # các điểm dừng
    heap = []                     # max-heap giả: chứa (-height, right)
    i, n = 0, len(buildings)      # i = chỉ số tòa tiếp theo chưa được cho vào
    res, prev = [], 0
    for x in xs:
        # Bước 1: cho vào mọi tòa bắt đầu tại x
        while i < n and buildings[i][0] <= x:
            l, r, h = buildings[i]
            heapq.heappush(heap, (-h, r))
            i += 1
        # Bước 2: xóa lười — vứt đỉnh đã hết hạn (nửa khoảng: r <= x là đã hết)
        while heap and heap[0][1] <= x:
            heapq.heappop(heap)
        # Bước 3: đọc chiều cao hiện tại (heap rỗng = mặt đất)
        cur = -heap[0][0] if heap else 0
        # Bước 4: nén — chỉ ghi khi chiều cao đổi
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


print(skyline_heap(ex1))
print(skyline_heap(ex2))
assert skyline_heap(ex1) == out1 == skyline_bruteforce(ex1)
assert skyline_heap(ex2) == out2 == skyline_bruteforce(ex2)
print("Khớp output của đề và khớp brute force!")

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
[[0, 3], [5, 0]]
Khớp output của đề và khớp brute force!


### 5.3 Trace bằng tay — Example 1

Ghi tên tòa kèm chiều cao cho dễ đọc (heap thật chứa `(-h, r)`). Cột "Heap sau bước" liệt kê từ đỉnh xuống; "(hết)" là rác đã hết hạn nhưng chưa bị vứt.

| x | Push (bắt đầu tại x) | Pop khi dọn đỉnh | Heap sau bước | cur | prev | Hành động |
|---:|---|---|---|---:|---:|---|
| 2 | A (h=10, r=9) | — | A10 | 10 | 0 | ghi `[2, 10]` |
| 3 | B (h=15, r=7) | — | B15, A10 | 15 | 10 | ghi `[3, 15]` |
| 5 | C (h=12, r=12) | — | B15, C12, A10 | 15 | 15 | bỏ qua (C bị B che) |
| 7 | — | B (7 ≤ 7) | C12, A10 | 12 | 15 | ghi `[7, 12]` |
| 9 | — | — (A hết hạn nhưng **không** ở đỉnh) | C12, A10(hết) | 12 | 12 | bỏ qua |
| 12 | — | C (12 ≤ 12), rồi A (9 ≤ 12) | (rỗng) | 0 | 12 | ghi `[12, 0]` |
| 15 | D (h=10, r=20) | — | D10 | 10 | 0 | ghi `[15, 10]` |
| 19 | E (h=8, r=24) | — | D10, E8 | 10 | 10 | bỏ qua |
| 20 | — | D (20 ≤ 20) | E8 | 8 | 10 | ghi `[20, 8]` |
| 24 | — | E (24 ≤ 24) | (rỗng) | 0 | 8 | ghi `[24, 0]` |

Hai dòng đáng chú ý nhất: **x = 9** (xóa lười: A đã hết nhưng cứ để đó) và **x = 12** (một lần dọn vứt liên tiếp 2 tòa rồi chạm mặt đất).

Hàm `trace_skyline_heap` dưới đây chạy y hệt `skyline_heap` nhưng đẩy thêm tên tòa vào tuple `(-h, r, tên)` để in ra.

In [15]:
def trace_skyline_heap(buildings, names="ABCDEFGHIJ"):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})
    heap = []                                   # (-h, r, tên)
    i, n = 0, len(buildings)
    res, prev = [], 0
    for x in xs:
        pushed, popped = [], []
        while i < n and buildings[i][0] <= x:
            l, r, h = buildings[i]
            heapq.heappush(heap, (-h, r, names[i]))
            pushed.append(names[i])
            i += 1
        while heap and heap[0][1] <= x:
            popped.append(heapq.heappop(heap)[2])
        cur = -heap[0][0] if heap else 0
        inside = ", ".join(f"{nm}{-nh}" + ("(hết)" if r <= x else "") for nh, r, nm in sorted(heap)) or "(rỗng)"
        action = f"ghi [{x}, {cur}]" if cur != prev else "bỏ qua"
        print(f"x={x:>2} | push {','.join(pushed) or '-':<3} | pop {','.join(popped) or '-':<3} "
              f"| heap: {inside:<17} | cur={cur:>2} prev={prev:>2} -> {action}")
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


result = trace_skyline_heap(ex1)
print("Kết quả:", result)
assert result == out1

x= 2 | push A   | pop -   | heap: A10               | cur=10 prev= 0 -> ghi [2, 10]
x= 3 | push B   | pop -   | heap: B15, A10          | cur=15 prev=10 -> ghi [3, 15]
x= 5 | push C   | pop -   | heap: B15, C12, A10     | cur=15 prev=15 -> bỏ qua
x= 7 | push -   | pop B   | heap: C12, A10          | cur=12 prev=15 -> ghi [7, 12]
x= 9 | push -   | pop -   | heap: C12, A10(hết)     | cur=12 prev=12 -> bỏ qua
x=12 | push -   | pop C,A | heap: (rỗng)            | cur= 0 prev=12 -> ghi [12, 0]
x=15 | push D   | pop -   | heap: D10               | cur=10 prev= 0 -> ghi [15, 10]
x=19 | push E   | pop -   | heap: D10, E8           | cur=10 prev=10 -> bỏ qua
x=20 | push -   | pop D   | heap: E8                | cur= 8 prev=10 -> ghi [20, 8]
x=24 | push -   | pop E   | heap: (rỗng)            | cur= 0 prev= 8 -> ghi [24, 0]
Kết quả: [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]


Trace thứ hai với một input "khó" hơn, gom các tình huống trùng tọa độ:

- A = `[1,3,4]` và B = `[1,8,2]` **cùng bắt đầu** tại x = 1 → phải push cả hai rồi mới đọc chiều cao.
- C = `[3,5,4]` bắt đầu đúng lúc A kết thúc, **cùng chiều cao 4**; D = `[5,6,4]` cũng vậy với C → đường viền cao 4 liền một mạch từ 1 tới 6, **không** được ghi `[3, 4]` hay `[5, 4]`.
- Khi hai tuple cùng `-h`, tuple có `r` nhỏ hơn đứng trên (so phần tử thứ hai) → tòa hết hạn sớm hơn được dọn trước, rất tự nhiên.
- Tại x = 6, D kết thúc và tòa thấp B (cao 2, bị che suốt từ đầu) mới lộ ra → `[6, 2]`.

Dự đoán trước: kết quả phải là `[[1, 4], [6, 2], [8, 0]]`.

In [16]:
tricky_heap = [[1, 3, 4], [1, 8, 2], [3, 5, 4], [5, 6, 4]]
result = trace_skyline_heap(tricky_heap)
print("Kết quả:", result)
assert result == skyline_bruteforce(tricky_heap) == [[1, 4], [6, 2], [8, 0]]

x= 1 | push A,B | pop -   | heap: A4, B2            | cur= 4 prev= 0 -> ghi [1, 4]
x= 3 | push C   | pop A   | heap: C4, B2            | cur= 4 prev= 4 -> bỏ qua
x= 5 | push D   | pop C   | heap: D4, B2            | cur= 4 prev= 4 -> bỏ qua
x= 6 | push -   | pop D   | heap: B2                | cur= 2 prev= 4 -> ghi [6, 2]
x= 8 | push -   | pop B   | heap: (rỗng)            | cur= 0 prev= 2 -> ghi [8, 0]
Kết quả: [[1, 4], [6, 2], [8, 0]]


## 6. Code nộp LeetCode (cách chính)

Chuyển `skyline_heap` thành method của `class Solution` với đúng signature của đề. Vài ghi chú:

- Trên LeetCode, `List` đã có sẵn và các module chuẩn như `heapq` thường được import tự động; trong notebook ta tự import. Ghi `import heapq` ở đầu vẫn an toàn.
- Ta dựa vào constraint "`buildings` đã sắp theo `left`" nên không sort lại input.
- Không sửa input (`buildings`), chỉ tạo list mới cho kết quả.

Bộ test dưới gồm 2 ví dụ của đề và các edge case dễ sai: 1 tòa, tòa lồng trong tòa, cùng `left`, hai tòa giống hệt, có khoảng trống, chạm nhau khác chiều cao, bậc thang đi xuống, cùng `right`, tọa độ cực lớn `2^31 - 1`.

In [17]:
import heapq
from typing import List


class Solution:
    def getSkyline(self, buildings: List[List[int]]) -> List[List[int]]:
        xs = sorted({x for l, r, _ in buildings for x in (l, r)})
        heap = []                       # (-height, right)
        i, n = 0, len(buildings)
        res, prev = [], 0
        for x in xs:
            while i < n and buildings[i][0] <= x:
                l, r, h = buildings[i]
                heapq.heappush(heap, (-h, r))
                i += 1
            while heap and heap[0][1] <= x:
                heapq.heappop(heap)
            cur = -heap[0][0] if heap else 0
            if cur != prev:
                res.append([x, cur])
                prev = cur
        return res


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


BIG = 2**31 - 1
tests = [
    ("Example 1", ex1, out1),
    ("Example 2", ex2, out2),
    ("1 tòa", [[1, 2, 1]], [[1, 1], [2, 0]]),
    ("tòa lồng trong tòa", [[1, 10, 5], [2, 3, 8]], [[1, 5], [2, 8], [3, 5], [10, 0]]),
    ("cùng left, khác cao", [[1, 5, 3], [1, 5, 7]], [[1, 7], [5, 0]]),
    ("2 tòa giống hệt", [[1, 5, 3], [1, 5, 3]], [[1, 3], [5, 0]]),
    ("có khoảng trống", [[1, 2, 1], [3, 4, 1]], [[1, 1], [2, 0], [3, 1], [4, 0]]),
    ("chạm nhau, khác cao", [[0, 2, 3], [2, 5, 4]], [[0, 3], [2, 4], [5, 0]]),
    ("bậc thang đi xuống", [[0, 3, 3], [1, 5, 2]], [[0, 3], [3, 2], [5, 0]]),
    ("cùng right", [[0, 5, 2], [3, 5, 4]], [[0, 2], [3, 4], [5, 0]]),
    ("tọa độ cực lớn", [[0, BIG, BIG]], [[0, BIG], [BIG, 0]]),
]
sol = Solution()
for name, b, expected in tests:
    check(name, sol.getSkyline(b), expected)
print("Tất cả test của class Solution đều PASS!")

Example 1              PASS  [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
Example 2              PASS  [[0, 3], [5, 0]]
1 tòa                  PASS  [[1, 1], [2, 0]]
tòa lồng trong tòa     PASS  [[1, 5], [2, 8], [3, 5], [10, 0]]
cùng left, khác cao    PASS  [[1, 7], [5, 0]]
2 tòa giống hệt        PASS  [[1, 3], [5, 0]]
có khoảng trống        PASS  [[1, 1], [2, 0], [3, 1], [4, 0]]
chạm nhau, khác cao    PASS  [[0, 3], [2, 4], [5, 0]]
bậc thang đi xuống     PASS  [[0, 3], [3, 2], [5, 0]]
cùng right             PASS  [[0, 2], [3, 4], [5, 0]]
tọa độ cực lớn         PASS  [[0, 2147483647], [2147483647, 0]]
Tất cả test của class Solution đều PASS!


## 7. Cách 3 — Sweep line theo **sự kiện** và quy tắc sắp xếp

Cách 2 dừng ở từng tọa độ x và **gom** mọi thay đổi tại x. Một phiên bản rất phổ biến trên mạng làm khác: biến mỗi tòa `[l, r, h]` thành **2 sự kiện**, sắp xếp tất cả sự kiện, rồi xử lý **từng sự kiện một**:

- sự kiện **VÀO** tại `x = l`: thêm chiều cao `h` vào tập đang đứng;
- sự kiện **RA** tại `x = r`: bỏ chiều cao `h` khỏi tập đang đứng.

Sau **mỗi** sự kiện, đọc chiều cao lớn nhất và ghi key point nếu nó đổi. Vì không gom theo x, khi nhiều sự kiện có **cùng x**, **thứ tự xử lý quyết định đúng sai**. Cần đúng 3 quy tắc:

| Quy tắc (khi cùng x) | Nếu làm ngược thì sao? (ví dụ) |
|---|---|
| 1. **VÀO trước RA** | `[0,2,3]` ra và `[2,5,3]` vào tại x = 2. Nếu ra trước: chiều cao tụt về 0 rồi lên lại 3 → ghi thừa `[2,0]`, `[2,3]`. |
| 2. Giữa các sự kiện VÀO: **cao trước thấp** | `[1,5,3]` và `[1,5,7]` cùng vào tại x = 1. Nếu thấp trước: ghi `[1,3]` rồi `[1,7]` — hai điểm cùng x. |
| 3. Giữa các sự kiện RA: **thấp trước cao** | `[1,5,3]`, `[1,5,7]` cùng ra tại x = 5, bên dưới còn `[0,9,1]`. Nếu cao ra trước: ghi `[5,3]` (tòa 3 "lộ" ra trong chốc lát) rồi `[5,1]`. |

**Mẹo mã hóa** để `sort()` mặc định tự làm đúng cả 3 quy tắc: **VÀO = `(l, -h)`**, **RA = `(r, h)`**.

- Cùng x: VÀO mang số âm, RA mang số dương, âm < dương → **VÀO trước RA**. ✔
- Hai sự kiện VÀO: `-15 < -10` → tòa cao 15 đứng trước → **cao trước thấp**. ✔
- Hai sự kiện RA: `10 < 15` → tòa cao 10 ra trước → **thấp trước cao**. ✔

Input `tricky` dưới đây gom đủ cả 3 tình huống: tại x = 1 hai tòa cùng vào (8 và 5); tại x = 4 một tòa cao 8 vào trong khi hai tòa (5 và 8) ra; tại x = 6 hai tòa cùng ra (3 và 8). Hãy xem `sort()` xếp chúng thế nào.

In [18]:
tricky = [[1, 4, 5], [1, 4, 8], [2, 6, 3], [4, 6, 8], [5, 9, 2]]

events = []
for l, r, hh in tricky:
    events.append((l, -hh))      # VÀO: số âm
    events.append((r, hh))       # RA : số dương
events.sort()

for x, v in events:
    kind = "VÀO" if v < 0 else "RA "
    print(f"x = {x}: {kind} cao {abs(v)}   (tuple {(x, v)})")

x = 1: VÀO cao 8   (tuple (1, -8))
x = 1: VÀO cao 5   (tuple (1, -5))
x = 2: VÀO cao 3   (tuple (2, -3))
x = 4: VÀO cao 8   (tuple (4, -8))
x = 4: RA  cao 5   (tuple (4, 5))
x = 4: RA  cao 8   (tuple (4, 8))
x = 5: VÀO cao 2   (tuple (5, -2))
x = 6: RA  cao 3   (tuple (6, 3))
x = 6: RA  cao 8   (tuple (6, 8))
x = 9: RA  cao 2   (tuple (9, 2))


### 7.1 Sự kiện RA chỉ biết chiều cao → xóa lười bằng `Counter`

Ở Cách 2, heap chứa `(-h, r)` nên tự biết tòa nào hết hạn. Ở đây heap chỉ chứa con số `-h`. Khi gặp sự kiện RA "một tòa cao 8 kết thúc", ta không biết nó nằm đâu trong heap → lại **xóa lười**, nhưng ghi sổ "nợ" bằng một **`Counter`**:

- `to_remove[8] += 1` nghĩa là: đang **nợ** việc xóa một phần tử cao 8.
- Mỗi lần đọc đỉnh: chừng nào đỉnh đang bị nợ (`to_remove[-heap[0]] > 0`) thì pop nó và trừ nợ đi 1.

Vì heap chỉ lưu con số, xóa **bản sao nào** của số 8 cũng như nhau → ghi sổ theo chiều cao là đúng.

`Counter` (trong module `collections`) là một `dict` chuyên dùng để **đếm**: đọc một khóa chưa có sẽ trả về `0` thay vì báo lỗi `KeyError`, nên viết `to_remove[h] += 1` không cần kiểm tra trước.

Ta đặt sẵn số `0` (mặt đất) trong heap: không tòa nào cao 0 nên nó không bao giờ bị xóa → heap **không bao giờ rỗng** → không cần `if heap`.

In [19]:
from collections import Counter

to_remove = Counter()
print(to_remove[8])                  # 0: khóa chưa có vẫn đọc được
to_remove[8] += 1
to_remove[8] += 1
to_remove[3] += 1
print(to_remove)                     # đếm: số 8 nợ 2 lần, số 3 nợ 1 lần
to_remove[8] -= 1
print(to_remove[8], to_remove[3], to_remove[100])

0
Counter({8: 2, 3: 1})
1 1 0


In [20]:
def skyline_events(buildings):
    events = []
    for l, r, h in buildings:
        events.append((l, -h))          # VÀO
        events.append((r, h))           # RA
    events.sort()                       # tự động: VÀO trước RA, VÀO cao trước, RA thấp trước
    heap = [0]                          # chứa -height; 0 = mặt đất, không bao giờ bị xóa
    to_remove = Counter()               # sổ nợ: chiều cao -> số bản sao cần xóa
    res, prev = [], 0
    for x, h in events:
        if h < 0:
            heapq.heappush(heap, h)     # h đã là -height
        else:
            to_remove[h] += 1           # ghi nợ, CHƯA xóa
        while to_remove[-heap[0]] > 0:  # đỉnh đang bị nợ -> xóa thật
            to_remove[-heap[0]] -= 1
            heapq.heappop(heap)
        cur = -heap[0]
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


for b in [ex1, ex2, tricky]:
    assert skyline_events(b) == skyline_bruteforce(b)
print(skyline_events(ex1))
print(skyline_events(tricky))

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
[[1, 8], [6, 2], [9, 0]]


In [21]:
def trace_skyline_events(buildings):
    events = sorted([(l, -h) for l, r, h in buildings] + [(r, h) for l, r, h in buildings])
    heap, to_remove = [0], Counter()
    res, prev = [], 0
    for x, h in events:
        if h < 0:
            heapq.heappush(heap, h)
            kind = f"VÀO {-h}"
        else:
            to_remove[h] += 1
            kind = f"RA  {h}"
        while to_remove[-heap[0]] > 0:
            to_remove[-heap[0]] -= 1
            heapq.heappop(heap)
        cur = -heap[0]
        heights = sorted((-v for v in heap), reverse=True)        # in từ cao xuống thấp
        debt = {k: v for k, v in to_remove.items() if v > 0}
        action = f"ghi [{x}, {cur}]" if cur != prev else "bỏ qua"
        print(f"x={x} {kind:<6} | heap (cao): {str(heights):<15} | nợ: {str(debt):<14} | cur={cur} -> {action}")
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


result = trace_skyline_events(tricky)
print("Kết quả:", result)

x=1 VÀO 8  | heap (cao): [8, 0]          | nợ: {}             | cur=8 -> ghi [1, 8]
x=1 VÀO 5  | heap (cao): [8, 5, 0]       | nợ: {}             | cur=8 -> bỏ qua
x=2 VÀO 3  | heap (cao): [8, 5, 3, 0]    | nợ: {}             | cur=8 -> bỏ qua
x=4 VÀO 8  | heap (cao): [8, 8, 5, 3, 0] | nợ: {}             | cur=8 -> bỏ qua
x=4 RA  5  | heap (cao): [8, 8, 5, 3, 0] | nợ: {5: 1}         | cur=8 -> bỏ qua
x=4 RA  8  | heap (cao): [8, 5, 3, 0]    | nợ: {5: 1}         | cur=8 -> bỏ qua
x=5 VÀO 2  | heap (cao): [8, 5, 3, 2, 0] | nợ: {5: 1}         | cur=8 -> bỏ qua
x=6 RA  3  | heap (cao): [8, 5, 3, 2, 0] | nợ: {5: 1, 3: 1}   | cur=8 -> bỏ qua
x=6 RA  8  | heap (cao): [2, 0]          | nợ: {}             | cur=2 -> ghi [6, 2]
x=9 RA  2  | heap (cao): [0]             | nợ: {}             | cur=0 -> ghi [9, 0]
Kết quả: [[1, 8], [6, 2], [9, 0]]


Đọc trace trên:

- **x = 1**: VÀO 8 được xử lý trước VÀO 5 → chỉ ghi `[1, 8]` một lần (quy tắc 2).
- **x = 4**: VÀO 8 (tòa `[4,6,8]`) được xử lý **trước** hai sự kiện RA → chiều cao không hề tụt → không ghi gì (quy tắc 1). RA 5 chỉ ghi nợ vì 5 không ở đỉnh; RA 8 xóa **một** bản sao 8, vẫn còn bản 8 của tòa mới.
- **x = 6**: RA 3 đi trước (quy tắc 3) — chỉ ghi nợ; rồi RA 8 xóa đỉnh 8, đỉnh mới là 5 (đang nợ từ x = 4) → xóa; đỉnh 3 (đang nợ) → xóa; đỉnh 2 sạch → ghi `[6, 2]` đúng **một** lần.

Để ý số 5 nằm "nợ" trong heap từ x = 4 tới x = 6 — y hệt "rác" của Cách 2.

### 7.2 Bản "gọn" hay gặp: sự kiện mang theo `right`

Kết hợp ý của Cách 2 (heap chứa `(-h, r)`, dọn theo hạn `r`) với danh sách sự kiện:

- sự kiện VÀO: `(l, -h, r)` — mang theo hạn `r` để push vào heap;
- sự kiện RA: `(r, 0, 0)` — không cần chiều cao; nó chỉ là một **điểm dừng** để đường quét dọn đỉnh tại `r`.

Sắp tuple 3 phần tử: cùng x thì so `-h` → VÀO (âm) trước RA (0), VÀO cao trước VÀO thấp — đúng quy tắc 1 và 2. Quy tắc 3 không cần nữa, vì ngay **sự kiện đầu tiên** tại x đã dọn sạch mọi đỉnh hết hạn (`r <= x`).

Hai **lính canh** (sentinel) giúp code không cần `if`:

- `heap = [(0, float("inf"))]`: mặt đất cao 0, hạn **vô cực** → không bao giờ bị pop, heap không bao giờ rỗng. (`float("inf")` lớn hơn mọi số nên `inf <= x` luôn False.)
- `res = [[0, 0]]`: một điểm giả cao 0 để so `res[-1][1]` ngay từ đầu; cuối cùng trả `res[1:]` để bỏ nó đi.

In [22]:
def skyline_events_compact(buildings):
    events = sorted([(l, -h, r) for l, r, h in buildings] + [(r, 0, 0) for _, r, _ in buildings])
    res = [[0, 0]]                       # lính canh: điểm giả cao 0
    heap = [(0, float("inf"))]           # lính canh: mặt đất, không bao giờ hết hạn
    for x, neg_h, r in events:
        while heap[0][1] <= x:           # dọn đỉnh hết hạn
            heapq.heappop(heap)
        if neg_h:                        # sự kiện VÀO (neg_h < 0)
            heapq.heappush(heap, (neg_h, r))
        if res[-1][1] != -heap[0][0]:    # chiều cao đổi -> ghi
            res.append([x, -heap[0][0]])
    return res[1:]


for b in [ex1, ex2, tricky, tricky_heap]:
    assert skyline_events_compact(b) == skyline_bruteforce(b)
print(skyline_events_compact(ex1))
print("Bản gọn khớp brute force trên 4 input!")

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
Bản gọn khớp brute force trên 4 input!


## 8. Cách 4 — Chia để trị (divide and conquer)

### 8.1 Nhắc nhanh: đệ quy và merge sort

- **Đệ quy** (recursion): một hàm gọi lại chính nó với bài toán **nhỏ hơn**, và phải có **trường hợp cơ sở** (base case) để dừng.
- **Merge sort**: chia mảng làm đôi → sort từng nửa bằng đệ quy → **merge** (trộn) hai nửa đã sort bằng hai con trỏ `i`, `j`: mỗi bước lấy phần tử nhỏ hơn trong `a[i]`, `b[j]`.

Skyline có cấu trúc y hệt merge sort:

| Merge sort | Skyline |
|---|---|
| Base case: mảng 1 phần tử đã sort sẵn | Base case: 1 tòa `[l, r, h]` → skyline `[[l, h], [r, 0]]` |
| Chia mảng làm đôi | Chia `buildings` làm đôi |
| Merge hai mảng đã sort | **Merge hai skyline** |

Cell dưới chạy merge sort trên `[5, 2, 4, 1, 3]` và in mỗi lần merge (thụt lề theo độ sâu đệ quy) để thấy rõ "chia nhỏ rồi gộp lên".

In [23]:
def merge_sorted(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i])
            i += 1
        else:
            out.append(b[j])
            j += 1
    return out + a[i:] + b[j:]          # phần còn dư của một bên


def merge_sort(arr, depth=0):
    if len(arr) <= 1:                   # base case
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid], depth + 1)
    right = merge_sort(arr[mid:], depth + 1)
    merged = merge_sorted(left, right)
    print("    " * depth + f"merge {left} + {right} -> {merged}")
    return merged


print("Kết quả:", merge_sort([5, 2, 4, 1, 3]))

    merge [5] + [2] -> [2, 5]
        merge [1] + [3] -> [1, 3]
    merge [4] + [1, 3] -> [1, 3, 4]
merge [2, 5] + [1, 3, 4] -> [1, 2, 3, 4, 5]
Kết quả: [1, 2, 3, 4, 5]


### 8.2 Merge hai skyline

Cho skyline trái `L` và skyline phải `R`. Hãy đọc chúng như hai hàm bậc thang `h1(x)` và `h2(x)`: chiều cao của một skyline tại x là `y` của **key point cuối cùng có tọa độ ≤ x**. Skyline gộp có chiều cao `max(h1(x), h2(x))`.

Dùng hai con trỏ `i`, `j` và hai biến `h1`, `h2` = chiều cao **hiện tại** của mỗi bên (ban đầu 0):

1. So `L[i][0]` với `R[j][0]`, lấy điểm có x **nhỏ hơn**; cập nhật `h1` (nếu điểm đến từ L) hoặc `h2` (nếu từ R).
2. **Nếu hai x bằng nhau → cập nhật cả `h1` lẫn `h2` rồi mới tính** (cùng tinh thần "gom theo x" của Cách 2).
3. `cur = max(h1, h2)`; nếu `cur != prev` → ghi `[x, cur]`.
4. Một bên hết điểm thì tiếp tục với bên còn lại; chiều cao bên đã hết giữ nguyên bằng 0 (vì điểm cuối của mọi skyline có `y = 0`).

Ví dụ: lần merge cuối cùng khi chạy chia để trị trên Example 1 (chia ABCDE thành AB | CDE):

- `L` = skyline của A, B = `[[2,10],[3,15],[7,10],[9,0]]`
- `R` = skyline của C, D, E = `[[5,12],[12,0],[15,10],[20,8],[24,0]]`

| Bước | Lấy điểm | h1 | h2 | cur = max | Hành động |
|---:|---|---:|---:|---:|---|
| 1 | L `[2,10]` | 10 | 0 | 10 | ghi `[2,10]` |
| 2 | L `[3,15]` | 15 | 0 | 15 | ghi `[3,15]` |
| 3 | R `[5,12]` | 15 | 12 | 15 | bỏ qua |
| 4 | L `[7,10]` | 10 | 12 | 12 | ghi `[7,12]` |
| 5 | L `[9,0]` | 0 | 12 | 12 | bỏ qua |
| 6 | R `[12,0]` | 0 | 0 | 0 | ghi `[12,0]` |
| 7 | R `[15,10]` | 0 | 10 | 10 | ghi `[15,10]` |
| 8 | R `[20,8]` | 0 | 8 | 8 | ghi `[20,8]` |
| 9 | R `[24,0]` | 0 | 0 | 0 | ghi `[24,0]` |

Bước 4 là chỗ hay: L tụt từ 15 xuống 10 nhưng R đang cao 12 → skyline gộp chỉ tụt xuống 12.

In [24]:
def merge_skylines(left, right):
    i = j = 0
    h1 = h2 = 0                  # chiều cao hiện tại của skyline trái / phải
    res, prev = [], 0
    while i < len(left) or j < len(right):
        if j == len(right) or (i < len(left) and left[i][0] < right[j][0]):
            x, h1 = left[i]      # điểm của bên trái đến trước
            i += 1
        elif i == len(left) or right[j][0] < left[i][0]:
            x, h2 = right[j]     # điểm của bên phải đến trước
            j += 1
        else:                    # trùng x: cập nhật CẢ HAI rồi mới tính
            x, h1 = left[i]
            h2 = right[j][1]
            i += 1
            j += 1
        cur = max(h1, h2)
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


def skyline_divide_conquer(buildings):
    def solve(lo, hi):           # skyline của buildings[lo:hi], luôn có hi - lo >= 1
        if hi - lo == 1:         # base case: 1 tòa
            l, r, h = buildings[lo]
            return [[l, h], [r, 0]]
        mid = (lo + hi) // 2
        return merge_skylines(solve(lo, mid), solve(mid, hi))
    return solve(0, len(buildings))


print(skyline_divide_conquer(ex1))
print(skyline_divide_conquer(ex2))
assert skyline_divide_conquer(ex1) == out1
assert skyline_divide_conquer(ex2) == out2

[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
[[0, 3], [5, 0]]


Hàm `solve` được định nghĩa **bên trong** `skyline_divide_conquer` (hàm lồng — nested function), nên nó dùng được biến `buildings` của hàm ngoài mà không cần truyền vào. Ta truyền chỉ số `lo`, `hi` thay vì cắt list `buildings[lo:hi]` để khỏi tốn công copy.

Cell dưới in (1) cây đệ quy trên Example 1 và (2) trace lần merge cuối — khớp bảng ở 8.2 — rồi (3) trace merge của Example 2, nơi hai skyline có điểm **trùng x = 2**: nhánh "trùng x" cập nhật cả `h1 = 0` lẫn `h2 = 3` cùng lúc nên chiều cao gộp vẫn là 3, không ghi gì.

In [25]:
def trace_divide_conquer(buildings, names="ABCDEFGHIJ"):
    def solve(lo, hi, depth):
        pad = "    " * depth
        if hi - lo == 1:
            l, r, h = buildings[lo]
            print(f"{pad}{names[lo]}: 1 tòa -> {[[l, h], [r, 0]]}")
            return [[l, h], [r, 0]]
        mid = (lo + hi) // 2
        sky = merge_skylines(solve(lo, mid, depth + 1), solve(mid, hi, depth + 1))
        print(f"{pad}merge {names[lo:mid]} + {names[mid:hi]} -> {sky}")
        return sky
    return solve(0, len(buildings), 0)


def trace_merge(left, right):
    i = j = 0
    h1 = h2 = 0
    res, prev, step = [], 0, 1
    while i < len(left) or j < len(right):
        if j == len(right) or (i < len(left) and left[i][0] < right[j][0]):
            x, h1 = left[i]
            i += 1
            src = f"L {[x, h1]}"
        elif i == len(left) or right[j][0] < left[i][0]:
            x, h2 = right[j]
            j += 1
            src = f"R {[x, h2]}"
        else:
            x, h1 = left[i]
            h2 = right[j][1]
            i += 1
            j += 1
            src = f"L+R x={x}"
        cur = max(h1, h2)
        action = f"ghi [{x}, {cur}]" if cur != prev else "bỏ qua"
        print(f"  bước {step}: lấy {src:<10} h1={h1:>2} h2={h2:>2} cur={cur:>2} -> {action}")
        if cur != prev:
            res.append([x, cur])
            prev = cur
        step += 1
    return res


print("(1) Cây đệ quy trên Example 1:")
trace_divide_conquer(ex1)
print("(2) Trace lần merge cuối AB + CDE:")
trace_merge(skyline_divide_conquer(ex1[:2]), skyline_divide_conquer(ex1[2:]))
print("(3) Trace merge Example 2 (trùng x = 2):")
print("  ->", trace_merge([[0, 3], [2, 0]], [[2, 3], [5, 0]]))

(1) Cây đệ quy trên Example 1:
        A: 1 tòa -> [[2, 10], [9, 0]]
        B: 1 tòa -> [[3, 15], [7, 0]]
    merge A + B -> [[2, 10], [3, 15], [7, 10], [9, 0]]
        C: 1 tòa -> [[5, 12], [12, 0]]
            D: 1 tòa -> [[15, 10], [20, 0]]
            E: 1 tòa -> [[19, 8], [24, 0]]
        merge D + E -> [[15, 10], [20, 8], [24, 0]]
    merge C + DE -> [[5, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
merge AB + CDE -> [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
(2) Trace lần merge cuối AB + CDE:
  bước 1: lấy L [2, 10]  h1=10 h2= 0 cur=10 -> ghi [2, 10]
  bước 2: lấy L [3, 15]  h1=15 h2= 0 cur=15 -> ghi [3, 15]
  bước 3: lấy R [5, 12]  h1=15 h2=12 cur=15 -> bỏ qua
  bước 4: lấy L [7, 10]  h1=10 h2=12 cur=12 -> ghi [7, 12]
  bước 5: lấy L [9, 0]   h1= 0 h2=12 cur=12 -> bỏ qua
  bước 6: lấy R [12, 0]  h1= 0 h2= 0 cur= 0 -> ghi [12, 0]
  bước 7: lấy R [15, 10] h1= 0 h2=10 cur=10 -> ghi [15, 10]
  bước 8: lấy R [20, 8]  h1= 0 h2= 8 cur= 8 -> ghi [20, 8]
  bước 9: lấ

### 8.3 Độ phức tạp của chia để trị

- Skyline của `k` tòa có **tối đa `2k`** key point (mỗi key point nằm ở một mép khác nhau). Nên merge hai skyline của hai nửa tốn `O(n)` ở mỗi tầng.
- Mỗi tầng chia đôi → có khoảng `log₂ n` tầng (với `n = 10^4` chỉ cỡ 14 tầng).
- Công thức truy hồi `T(n) = 2·T(n/2) + O(n)` → **Time `O(n log n)`**, giống hệt merge sort.
- **Space `O(n)`** cho các list trung gian + `O(log n)` cho call stack đệ quy.

So với heap: chia để trị **không cần cấu trúc dữ liệu đặc biệt** nào, chỉ cần hai con trỏ — và không cần `buildings` sắp sẵn. Đổi lại code dài hơn và tạo nhiều list trung gian.

## 9. Cách 5 (tùy chọn) — "Multiset" tự cài bằng `bisect`

Trong C++ / Java người ta hay giải bài này bằng `multiset` / `TreeMap`: một tập **có thứ tự, cho phép trùng**, xóa **thật** một phần tử bất kỳ và lấy max đều nhanh (`O(log k)`). Python chuẩn không có sẵn cấu trúc này. Trên LeetCode có thư viện ngoài `sortedcontainers` (lớp `SortedList`), nhưng notebook này chỉ dùng thư viện chuẩn, nên ta tự cài một bản đơn giản bằng module **`bisect`** (tìm kiếm nhị phân trên list **đã sắp xếp**):

- `bisect.insort(a, v)`: chèn `v` vào đúng chỗ để `a` vẫn tăng dần.
- `bisect.bisect_left(a, v)`: vị trí **đầu tiên** có giá trị `>= v` → nếu `v` có trong `a` thì đó là chỗ của một bản sao `v`, `a.pop(vị trí)` xóa đúng một bản.
- Max = `a[-1]` (phần tử cuối của list tăng dần).

Dùng multiset này với **đúng danh sách sự kiện và quy tắc sắp xếp của Cách 3**, chỉ khác: sự kiện RA **xóa thật** thay vì ghi nợ.

- Ưu điểm: trạng thái luôn "sạch", không cần hiểu xóa lười.
- Nhược điểm: chèn/xóa ở giữa list phải dịch các phần tử phía sau → `O(k)` mỗi lần (`k` = số tòa đang đứng) → worst case **`O(n²)`**, dù thực tế khá nhanh vì việc dịch bộ nhớ chạy bằng C. Dùng `SortedList` thật thì mỗi thao tác chỉ khoảng `O(log n)` → `O(n log n)`.

In [26]:
import bisect

a = [3, 8, 8]
bisect.insort(a, 5)
print(a)                            # [3, 5, 8, 8]
print(bisect.bisect_left(a, 8))     # 2: vị trí đầu tiên có giá trị >= 8
a.pop(bisect.bisect_left(a, 8))     # xóa đúng MỘT số 8
print(a, "max =", a[-1])


class SortedMultiset:
    # Tập có thứ tự, cho phép trùng; tự cài bằng list đã sắp + bisect
    def __init__(self):
        self.a = []

    def add(self, v):
        bisect.insort(self.a, v)            # tìm chỗ O(log k) + dịch phần tử O(k)

    def remove(self, v):
        self.a.pop(bisect.bisect_left(self.a, v))

    def get_max(self):
        return self.a[-1] if self.a else 0  # rỗng -> mặt đất


def skyline_multiset(buildings):
    events = sorted([(l, -h) for l, r, h in buildings] + [(r, h) for l, r, h in buildings])
    active = SortedMultiset()
    res, prev = [], 0
    for x, h in events:
        if h < 0:
            active.add(-h)
        else:
            active.remove(h)                # xóa THẬT, không cần sổ nợ
        cur = active.get_max()
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


for b in [ex1, ex2, tricky, tricky_heap]:
    assert skyline_multiset(b) == skyline_bruteforce(b)
print(skyline_multiset(ex1))
print("Cách 5 khớp brute force trên 4 input!")

[3, 5, 8, 8]
2
[3, 5, 8] max = 8
[[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
Cách 5 khớp brute force trên 4 input!


## 10. Kiểm thử chéo tất cả các cách

Ta sinh ngẫu nhiên nhiều input nhỏ **đúng constraints** (`l < r`, `h >= 1`, sắp theo `left`), tọa độ nhỏ để Bước 1a (tô từng ô) cũng chạy được và làm **trọng tài số 1**, chiều cao nhỏ (1..5) để có thật nhiều tình huống **trùng chiều cao / trùng mép** — đúng chỗ các cách dễ sai nhất. `random.seed(218)` cố định để lần chạy nào cũng ra cùng một bộ test.

Ngoài so khớp, ta kiểm thêm 3 tính chất mà mọi skyline hợp lệ phải có:

1. Tọa độ x **tăng ngặt** (không có 2 điểm cùng x).
2. Hai key point liên tiếp **khác chiều cao** (đã nén).
3. Điểm cuối có `y = 0`.

Cell thứ hai là một thí nghiệm tốc độ để **thấy tận mắt** khác biệt giữa `O(n²)` và `O(n log n)`: `n = 2000` tòa ngẫu nhiên với tọa độ tới cỡ 10^6, rồi `n = 10^4` (giới hạn của đề) cho các cách nhanh. Số giây cụ thể phụ thuộc máy, nhưng brute force sẽ chậm hơn các cách còn lại **hàng chục tới hàng trăm lần**.

In [27]:
import random


def random_buildings(max_n=7, max_x=15, max_h=5):
    n = random.randint(1, max_n)
    bs = []
    for _ in range(n):
        l = random.randint(0, max_x - 1)
        r = random.randint(l + 1, max_x)
        h = random.randint(1, max_h)
        bs.append([l, r, h])
    bs.sort()                      # sắp theo left (rồi tới right, height) như đề bảo đảm
    return bs


def is_valid_skyline(points):
    pairs = list(zip(points, points[1:]))
    return (all(a[0] < b[0] for a, b in pairs)          # 1. x tăng ngặt
            and all(a[1] != b[1] for a, b in pairs)     # 2. đã nén
            and points[-1][1] == 0)                     # 3. kết thúc ở mặt đất


methods = [skyline_bruteforce, skyline_heap, skyline_events, skyline_events_compact,
           skyline_divide_conquer, skyline_multiset, Solution().getSkyline]

random.seed(218)
for _ in range(2000):
    b = random_buildings()
    expected = skyline_grid(b)
    assert is_valid_skyline(expected)
    for f in methods:
        got = f(b)
        assert got == expected, (f.__name__, b, got, expected)
print(f"2000 bộ test ngẫu nhiên: {len(methods)} hàm khớp tuyệt đối với Bước 1a (tô từng ô)!")

2000 bộ test ngẫu nhiên: 7 hàm khớp tuyệt đối với Bước 1a (tô từng ô)!


In [28]:
import time


def make_city(n, seed):
    rng = random.Random(seed)          # bộ sinh số ngẫu nhiên riêng, không ảnh hưởng random.seed ở trên
    bs = []
    for _ in range(n):
        l = rng.randint(0, 10**6)
        bs.append([l, l + rng.randint(1, 10**5), rng.randint(1, 10**4)])
    bs.sort()
    return bs


def timed(f, b):
    t0 = time.perf_counter()
    out = f(b)
    return out, time.perf_counter() - t0


city = make_city(2000, seed=7)
reference, t_brute = timed(skyline_bruteforce, city)
print(f"n = 2000  | {'skyline_bruteforce':<24} {t_brute:7.3f} s  ({len(reference)} key point)")
for f in [skyline_heap, skyline_events, skyline_divide_conquer]:
    out, t = timed(f, city)
    assert out == reference
    print(f"n = 2000  | {f.__name__:<24} {t:7.3f} s")

city = make_city(10**4, seed=8)
outs = []
for f in [skyline_heap, skyline_events, skyline_divide_conquer]:
    out, t = timed(f, city)
    outs.append(out)
    print(f"n = 10000 | {f.__name__:<24} {t:7.3f} s")
assert outs[0] == outs[1] == outs[2]

n = 2000  | skyline_bruteforce         0.283 s  (43 key point)
n = 2000  | skyline_heap               0.002 s
n = 2000  | skyline_events             0.003 s
n = 2000  | skyline_divide_conquer     0.003 s
n = 10000 | skyline_heap               0.017 s
n = 10000 | skyline_events             0.016 s
n = 10000 | skyline_divide_conquer     0.017 s


## 11. Vì sao đúng? (invariant bằng lời)

### Cách 2 — sweep line + heap

Gọi các điểm dừng là `x_1 < x_2 < ... < x_K` (mọi `left`, `right`). **Bất biến** (invariant) đúng ngay sau khi xử lý xong mỗi điểm dừng `x`:

> **(I1)** Mọi tòa **đang phủ** `x` (tức `l <= x < r`) đều nằm trong heap.
>
> **(I2)** Nếu heap không rỗng, đỉnh heap là một tòa **còn hiệu lực** tại `x` (`r > x`).

- Vì sao (I1) đúng: một tòa được push đúng lúc đường quét tới `x = l` (nhờ con trỏ `i` và `buildings` sắp theo `left`), và chỉ bị pop khi nó ở đỉnh **với `r <= x`**, tức là khi nó **không còn** phủ x. Tòa đang phủ x thì chưa thể bị pop.
- Vì sao (I2) đúng: vòng `while heap and heap[0][1] <= x` chỉ dừng khi heap rỗng hoặc đỉnh có `r > x`.

Ghép lại: đỉnh heap lớn nhất trong **mọi** phần tử của heap, nên ≥ mọi tòa đang phủ x (do I1); bản thân nó cũng đang phủ x (đã push nên `l <= x`, và `r > x` do I2). Vậy `-heap[0][0]` **đúng bằng** `h(x)`. Nếu heap rỗng thì theo (I1) không tòa nào phủ x → `h(x) = 0`.

Cuối cùng, `h` không đổi trên mỗi đoạn `[x_k, x_{k+1})` (mục 3.2), nên dãy `h(x_1), ..., h(x_K)` mô tả **trọn vẹn** đường viền. Quy tắc `prev` chỉ ghi khi giá trị đổi → mỗi key point là đầu mút trái của một đoạn ngang **dài nhất có thể**, không có hai đoạn liên tiếp cùng chiều cao, và điểm cuối là `[x_K, 0]` (mục 3.3).

### Cách 3 — sự kiện + quy tắc sắp xếp

Sau khi xử lý **hết** các sự kiện tại x, heap (trừ đi phần đang nợ) đúng bằng tập chiều cao của các tòa phủ x, nên chiều cao đọc lúc đó là `h(x)`. Điều còn phải lo là các trạng thái **trung gian** giữa các sự kiện cùng x. Ba quy tắc bảo đảm chiều cao đọc được đổi **nhiều nhất một lần** tại mỗi x, và lần đổi đó đã là giá trị cuối:

- VÀO đi trước, cao trước: chỉ sự kiện VÀO đầu tiên (cao nhất) có thể làm chiều cao tăng. Nếu nó làm tăng thì tòa mới cao hơn mọi tòa cũ, còn các sự kiện RA phía sau đều là của tòa cũ → không chạm được tới đỉnh.
- RA đi sau, thấp trước: tòa thấp hơn đỉnh chỉ bị ghi nợ, không đổi chiều cao. Chiều cao chỉ tụt khi bản sao **cuối cùng** của chiều cao đỉnh bị xóa; vì đỉnh là lớn nhất và RA xếp tăng dần, lúc đó không còn sự kiện RA nào khác tại x → lần tụt xảy ra ở sự kiện cuối, đúng giá trị cuối.

### Cách 4 — chia để trị

Skyline của một nhóm tòa là đồ thị của `h(x)` trên nhóm đó; chia nhóm làm hai thì `h(x) = max(h_trái(x), h_phải(x))`. Trong `merge_skylines`, bất biến là: sau khi đọc xong mọi key point có tọa độ ≤ x của cả hai bên, `h1 = h_trái(x)` và `h2 = h_phải(x)`. Xử lý chung hai điểm trùng x bảo đảm ta chỉ tính `max(h1, h2)` khi đã cập nhật đủ cả hai → đúng tại mọi x, rồi nén bằng `prev` như mọi cách khác.

## 12. Độ phức tạp — so sánh các cách

Ký hiệu: `n` = số tòa; `W = max right - min left`. Output có tối đa `2n` key point.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1a. Tô từng ô | duyệt mọi số nguyên x, mỗi x duyệt n tòa | `O(n · W)` | `O(1)` thêm (ngoài output) | W tới ~2·10^9 → không dùng được |
| 1b. Tọa độ quan trọng | ≤ 2n điểm dừng, mỗi điểm duyệt n tòa | `O(n²)` | `O(n)` | "trọng tài" để test; n = 10^4 → TLE |
| **2. Sweep line + heap** (chính) | heap `(-h, r)`, gom theo x, xóa lười theo `r` | **`O(n log n)`** | `O(n)` | ngắn, dễ chứng minh, dùng constraint sắp sẵn |
| 3. Sự kiện + `Counter` | sort `(l,-h)` / `(r,h)`, xóa lười theo chiều cao | `O(n log n)` | `O(n)` | phải nhớ 3 quy tắc sắp xếp |
| 3'. Sự kiện bản gọn | sort `(l,-h,r)` / `(r,0,0)`, lính canh | `O(n log n)` | `O(n)` | ngắn nhất, hay gặp trên mạng |
| 4. Chia để trị | merge 2 skyline như merge sort | `O(n log n)` | `O(n)` + stack `O(log n)` | không cần heap, không cần input sắp sẵn |
| 5. Multiset tự cài | `bisect.insort` + xóa thật | `O(n²)` worst | `O(n)` | với `SortedList` thật → `O(n log n)` |

Vì sao Cách 2 là `O(n log n)`:

- Sắp xếp tối đa `2n` điểm dừng: `O(n log n)`.
- Mỗi tòa được push **1 lần** và pop **nhiều nhất 1 lần**, mỗi thao tác `O(log n)` → tổng `O(n log n)` — dù một vòng `while` đơn lẻ có thể pop nhiều phần tử (lập luận khấu hao ở mục 4.3).
- Phần còn lại tại mỗi điểm dừng là `O(1)`.

## 13. Lỗi thường gặp (code sai → vì sao → sửa)

| # | Lỗi | Hậu quả | Sửa |
|---|---|---|---|
| 1 | Coi tòa phủ cả mép phải: `l <= x <= r` (hoặc dọn đỉnh bằng `heap[0][1] < x`) | Tòa "sống" thêm đúng tại mép phải: Example 1 ra `[[2,10],[3,15],[9,12],[15,10],[24,8]]` — mất `[7,12]`, `[12,0]`, `[20,8]`, `[24,0]` | Dùng nửa khoảng `l <= x < r`; dọn đỉnh bằng `<=` |
| 2 | Quên nén (ghi mọi điểm dừng) | Có các đoạn liên tiếp cùng chiều cao như `[5,15]`, `[9,12]` → sai đề | So với `prev`, chỉ ghi khi chiều cao đổi |
| 3 | Đọc `heap[0]` khi heap rỗng | `IndexError` đúng chỗ mặt đất (x = 12 của Example 1) | `cur = -heap[0][0] if heap else 0`, hoặc đặt sẵn lính canh mặt đất |
| 4 | Mã hóa sự kiện "ngây thơ" `(x, h, "start")` / `(x, h, "end")` | Cùng x thì xếp theo `h` tăng dần và `"end" < "start"` theo bảng chữ cái → sai cả 3 quy tắc → nhiều điểm cùng x | Dùng `(l, -h)` và `(r, h)` |
| 5 | Merge: trùng x mà xử lý từng bên một | Điểm thừa `[2,0]`, `[2,3]` ở Example 2 | Trùng x → cập nhật cả `h1`, `h2` rồi mới tính |
| 6 | Max-heap quên đổi dấu khi đọc | Chiều cao âm kiểu `[2, -10]` | Luôn đọc `-heap[0][0]` |
| 7 | Bỏ tòa hết hạn bằng `heap.remove(...)` + `heapify` | Vẫn đúng nhưng `O(n)` mỗi lần → `O(n²)` | Xóa lười |

Ba cell dưới chạy thật các lỗi 1–5 để thấy output sai trông như thế nào.

In [29]:
# Lỗi 1: dùng đoạn đóng [l, r]
def skyline_closed_interval_wrong(buildings):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})
    res, prev = [], 0
    for x in xs:
        cur = 0
        for l, r, h in buildings:
            if l <= x <= r:                  # SAI: tại x = r tòa đã hết
                cur = max(cur, h)
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


# Lỗi 2: quên nén
def skyline_no_compress_wrong(buildings):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})
    return [[x, height_at(buildings, x)] for x in xs]   # SAI: ghi mọi điểm dừng


print("Đúng   :", out1)
print("Lỗi 1  :", skyline_closed_interval_wrong(ex1))
print("Lỗi 2  :", skyline_no_compress_wrong(ex1))

Đúng   : [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]
Lỗi 1  : [[2, 10], [3, 15], [9, 12], [15, 10], [24, 8]]
Lỗi 2  : [[2, 10], [3, 15], [5, 15], [7, 12], [9, 12], [12, 0], [15, 10], [19, 10], [20, 8], [24, 0]]


In [30]:
# Lỗi 3: quên trường hợp heap rỗng
def skyline_heap_no_guard_wrong(buildings):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})
    heap, i, res, prev = [], 0, [], 0
    for x in xs:
        while i < len(buildings) and buildings[i][0] <= x:
            heapq.heappush(heap, (-buildings[i][2], buildings[i][1]))
            i += 1
        while heap and heap[0][1] <= x:
            heapq.heappop(heap)
        cur = -heap[0][0]                    # SAI: heap có thể rỗng
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


try:
    skyline_heap_no_guard_wrong(ex1)
except IndexError as e:
    print("Lỗi 3 -> IndexError:", e)


# Lỗi 4: mã hóa sự kiện ngây thơ
def skyline_events_wrong_order(buildings):
    events = []
    for l, r, h in buildings:
        events.append((l, h, "start"))
        events.append((r, h, "end"))
    events.sort()                            # SAI: cùng x -> h tăng dần, "end" < "start"
    heap, to_remove, res, prev = [0], Counter(), [], 0
    for x, h, kind in events:
        if kind == "start":
            heapq.heappush(heap, -h)
        else:
            to_remove[h] += 1
        while to_remove[-heap[0]] > 0:
            to_remove[-heap[0]] -= 1
            heapq.heappop(heap)
        if -heap[0] != prev:
            prev = -heap[0]
            res.append([x, prev])
    return res


print("Đúng   :", skyline_events(tricky))
print("Lỗi 4  :", skyline_events_wrong_order(tricky))

Lỗi 3 -> IndexError: list index out of range
Đúng   : [[1, 8], [6, 2], [9, 0]]
Lỗi 4  : [[1, 5], [1, 8], [4, 3], [4, 8], [6, 2], [9, 0]]


In [31]:
# Lỗi 5: merge mà trùng x lại xử lý từng bên một
def merge_skylines_wrong(left, right):
    i = j = 0
    h1 = h2 = 0
    res, prev = [], 0
    while i < len(left) or j < len(right):
        if j == len(right) or (i < len(left) and left[i][0] <= right[j][0]):   # SAI: <= gộp luôn ca trùng
            x, h1 = left[i]
            i += 1
        else:
            x, h2 = right[j]
            j += 1
        cur = max(h1, h2)
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


L, R = [[0, 3], [2, 0]], [[2, 3], [5, 0]]       # hai nửa của Example 2
print("Đúng   :", merge_skylines(L, R))
print("Lỗi 5  :", merge_skylines_wrong(L, R))

Đúng   : [[0, 3], [5, 0]]
Lỗi 5  : [[0, 3], [2, 0], [2, 3], [5, 0]]


Đọc kết quả:

- **Lỗi 1**: mỗi tòa "sống" thêm đúng tại mép phải của nó → tại x = 7 vẫn thấy B (15) nên `[7,12]` bị lùi thành `[9,12]`; tại x = 12 vẫn thấy C, tại x = 20 vẫn thấy D, tại x = 24 vẫn thấy E → mất `[12,0]`, `[20,8]`, `[24,0]`, skyline **không bao giờ chạm đất** và kết thúc bằng điểm sai `[24,8]`.
- **Lỗi 2**: thừa `[5,15]`, `[9,12]`, `[19,10]` — đúng là các đoạn liên tiếp cùng chiều cao mà đề cấm.
- **Lỗi 3**: chương trình chết ngay khi chạm khoảng mặt đất giữa hai cụm tòa.
- **Lỗi 4**: có 2 điểm tại x = 1 và 2 điểm tại x = 4, trong đó `[4, 3]` là một cú "tụt rồi lên" không hề tồn tại.
- **Lỗi 5**: x = 2 sinh hai điểm `[2,0]`, `[2,3]` dù đường viền thật cao 3 liền mạch.

## 14. Mẹo nhận dạng pattern & biến thể

**Dấu hiệu "sweep line + heap":**

- Input là các **đoạn / khoảng** `[start, end)` trên một trục (thời gian, tọa độ...), mỗi đoạn kèm một giá trị (chiều cao, trọng số...).
- Câu hỏi dạng "tại mỗi thời điểm, **lớn nhất / nhỏ nhất** trong các đoạn **đang mở** là gì", hoặc hỏi **đường bao** (contour) của hợp các đoạn.
- Trục rất lớn (tới 10^9) → không duyệt từng điểm → chỉ dừng ở **mép** (tọa độ quan trọng / sự kiện).
- Cần bỏ phần tử "bất kỳ" khỏi heap → nghĩ ngay tới **xóa lười**: lưu kèm hạn dùng, hoặc ghi sổ nợ bằng `Counter`.

**Dấu hiệu "chia để trị":** kết quả của hai nửa có thể **gộp trong thời gian tuyến tính** (như merge hai list đã sắp) → `O(n log n)`.

**Bài liên quan nên luyện:**

| Bài | Liên hệ với Skyline |
|---|---|
| 56. Merge Intervals (Medium) | sắp theo mép trái, gộp các khoảng chồng nhau |
| 253. Meeting Rooms II (Medium, Premium) | số khoảng chồng nhau tối đa: sự kiện, hoặc min-heap theo giờ kết thúc |
| 1094. Car Pooling (Medium) | sự kiện lên/xuống xe, cộng dồn số khách |
| 732. My Calendar III (Hard) | số lịch chồng nhau tối đa sau mỗi lần thêm |
| 699. Falling Squares (Hard) | chiều cao chồng lên nhau trên trục x |
| 850. Rectangle Area II (Hard) | sweep line trên hình chữ nhật, nén tọa độ |
| 1851. Minimum Interval to Include Each Query (Hard) | sort + heap + xóa lười theo mép phải — gần như anh em sinh đôi |

**Biến thể nhỏ — thứ tự sự kiện phụ thuộc vào câu hỏi.** Bài "tối đa bao nhiêu tòa cùng phủ một điểm" (kiểu Meeting Rooms II) cũng dùng sự kiện, nhưng ta lấy max qua **mọi trạng thái trung gian**, nên trạng thái trung gian phải hợp lệ: với nửa khoảng, tại cùng x phải xử lý **RA trước VÀO** (tòa hết tại x không còn phủ x). Mã hóa `(l, +1)` và `(r, -1)` thì `sort()` tự xếp `-1` trước `+1`. Ngược hẳn với Skyline! Cell dưới minh họa.

In [32]:
def max_overlap(buildings):
    events = []
    for l, r, _ in buildings:
        events.append((l, +1))       # VÀO
        events.append((r, -1))       # RA: cùng x thì -1 < +1 -> RA trước
    events.sort()
    cnt = best = 0
    for x, d in events:
        cnt += d
        best = max(best, cnt)
    return best


print(max_overlap(ex1))   # 3: A, B, C cùng phủ đoạn [5, 7)
print(max_overlap(ex2))   # 1: [0,2) và [2,5) chỉ chạm nhau, không chồng lên nhau

3
1


## 15. Bài tập vận dụng

Làm theo thứ tự, từ dễ đến khó. Cell TODO chỉ định nghĩa hàm, test để sẵn ở dạng comment: tự làm xong rồi mới bỏ dấu `#` để chạy. Gợi ý ở mục 16, đáp án ở mục 17 (cuối notebook).

### Bài 1 — Đoán output (heap + xóa lười)

Không chạy code, hãy đoán 3 dòng in ra. Đặc biệt: vì sao dòng cuối in `len(heap)` lại **không** phải 1?

```python
import heapq
heap = []
for h, r in [(10, 9), (15, 7), (12, 12)]:
    heapq.heappush(heap, (-h, r))
print(heap[0])
print(-heap[0][0])

x = 9
while heap and heap[0][1] <= x:
    heapq.heappop(heap)
print(len(heap), -heap[0][0])
```

### Bài 2 — Tính tay

Cho `buildings = [[1,5,3],[2,4,6],[3,7,4]]`.

1. Liệt kê các tọa độ quan trọng và tính `h(x)` tại mỗi điểm.
2. Viết skyline (danh sách key point).
3. Lập bảng trace Cách 2 (cột: x, push, pop, heap sau bước, cur, hành động). Tại x = 5, tòa nào là "rác" còn nằm trong heap, và nó bị vứt ở điểm dừng nào?

### Bài 3 — Điền TODO: cách chính sweep line + heap

Hoàn thiện hàm theo 4 bước: cho vào → dọn đỉnh → đọc chiều cao → nén. Giả sử `buildings` đã sắp theo `left`.

In [33]:
def skyline_heap_exercise(buildings):
    xs = None          # TODO 1: tập mọi left/right, sắp tăng dần
    heap = []          # chứa (-height, right)
    i, n = 0, len(buildings)
    res, prev = [], 0
    # TODO 2: for x in xs:
    # TODO 3:     while i < n and buildings[i][0] <= x: push (-h, r), i += 1
    # TODO 4:     while heap and đỉnh đã hết hạn (right <= x): pop
    # TODO 5:     cur = chiều cao đỉnh (nhớ đổi dấu), hoặc 0 nếu heap rỗng
    # TODO 6:     nếu cur != prev: ghi [x, cur] và cập nhật prev
    pass


# Tự làm xong rồi mới mở test:
# assert skyline_heap_exercise(ex1) == out1
# assert skyline_heap_exercise(ex2) == out2
# assert skyline_heap_exercise([[1, 10, 5], [2, 3, 8]]) == [[1, 5], [2, 8], [3, 5], [10, 0]]
# assert skyline_heap_exercise([[1, 2, 1], [3, 4, 1]]) == [[1, 1], [2, 0], [3, 1], [4, 0]]

### Bài 4 — Tự viết lại: merge hai skyline

Không nhìn lại mục 8, tự viết hàm gộp hai skyline `left`, `right` (mỗi cái là list key point hợp lệ, có thể rỗng). Nhớ: hai con trỏ, hai chiều cao `h1`, `h2`, **trùng x thì cập nhật cả hai**, chỉ ghi khi `max(h1, h2)` đổi.

```python
merge_skylines_exercise([[2,10],[3,15],[7,10],[9,0]], [[5,12],[12,0]])  ->  [[2,10],[3,15],[7,12],[12,0]]
merge_skylines_exercise([[0,3],[2,0]], [[2,3],[5,0]])                   ->  [[0,3],[5,0]]
```

In [34]:
def merge_skylines_exercise(left, right):
    # TODO: hai con trỏ i, j; h1, h2; xử lý 3 trường hợp (L trước, R trước, trùng x)
    raise NotImplementedError


# Gợi ý test:
# assert merge_skylines_exercise([[2, 10], [3, 15], [7, 10], [9, 0]], [[5, 12], [12, 0]]) == [[2, 10], [3, 15], [7, 12], [12, 0]]
# assert merge_skylines_exercise([[0, 3], [2, 0]], [[2, 3], [5, 0]]) == [[0, 3], [5, 0]]
# assert merge_skylines_exercise([[1, 1], [2, 0]], [[3, 1], [4, 0]]) == [[1, 1], [2, 0], [3, 1], [4, 0]]
# assert merge_skylines_exercise([], [[1, 2], [3, 0]]) == [[1, 2], [3, 0]]

### Bài 5 — Biến thể: diện tích hình bóng

Tính **diện tích** phần hình bóng (hợp của mọi hình chữ nhật — phần chồng nhau chỉ tính một lần). Hãy tận dụng skyline: giữa hai key point liên tiếp `[x1, y1]` và `[x2, y2]` là một hình chữ nhật rộng `x2 - x1`, cao `y1`.

```python
skyline_area(ex1)                    -> 212
skyline_area(ex2)                    -> 15
skyline_area([[1,10,5],[2,3,8]])     -> 48
```

Tự kiểm tra Example 1 bằng tay: `1·10 + 4·15 + 5·12 + 3·0 + 5·10 + 4·8 = 212`.

In [35]:
def skyline_area(buildings):
    # TODO 1: lấy skyline (dùng Solution().getSkyline hoặc skyline_heap)
    # TODO 2: cộng (x2 - x1) * y1 cho mọi cặp key point liên tiếp
    pass


# Gợi ý test:
# assert skyline_area(ex1) == 212
# assert skyline_area(ex2) == 15
# assert skyline_area([[1, 10, 5], [2, 3, 8]]) == 48

### Bài 6 — Mini challenge: trả lời nhiều truy vấn chiều cao

Cho `buildings` và list `queries` gồm `q` tọa độ x (không cần sắp xếp). Trả về list `h(x)` cho từng truy vấn, theo đúng thứ tự `queries`.

Yêu cầu: **không** gọi `height_at` cho từng truy vấn (sẽ là `O(n·q)`). Hãy tính skyline một lần, rồi với mỗi truy vấn dùng `bisect` tìm **key point cuối cùng có tọa độ ≤ x** → tổng `O((n + q) log n)`.

```python
heights_at(ex1, [0, 2, 3, 6, 7, 11, 12, 14, 19, 23, 24, 100])
-> [0, 10, 15, 15, 12, 12, 0, 0, 10, 8, 0, 0]
```

In [36]:
def heights_at(buildings, queries):
    # TODO 1: sky = skyline của buildings; key_xs = [x for x, _ in sky]
    # TODO 2: với mỗi q: k = bisect.bisect_right(key_xs, q) - 1
    # TODO 3:     k < 0 (bên trái mọi tòa) -> 0; ngược lại -> sky[k][1]
    pass


# Gợi ý test:
# assert heights_at(ex1, [0, 2, 3, 6, 7, 11, 12, 14, 19, 23, 24, 100]) == [0, 10, 15, 15, 12, 12, 0, 0, 10, 8, 0, 0]
# assert heights_at(ex2, [2, 0, 5, 4]) == [3, 3, 0, 3]

## 16. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Heap chứa `(-h, r)`: đỉnh là tòa **cao nhất**. | Dọn đỉnh chỉ nhìn `heap[0]`, không nhìn phần tử khác. | Tại x = 9: đỉnh `(-15, 7)` hết hạn → pop; đỉnh mới `(-12, 12)` còn hạn → dừng. `(-10, 9)` vẫn nằm dưới. |
| 2 | Điểm dừng = mọi left/right. | `h(x)` = max các tòa có `l <= x < r`. | Tại x = 4, B (6) hết; còn A (3) và C (4) → 4. Tại x = 5, A hết nhưng nằm dưới C. |
| 3 | 4 bước: vào → dọn → đọc → nén. | `sorted({x for l, r, _ in buildings for x in (l, r)})`; `heapq.heappush(heap, (-h, r))`. | `while heap and heap[0][1] <= x: heapq.heappop(heap)`; `cur = -heap[0][0] if heap else 0`. |
| 4 | Giống merge hai list đã sắp, nhưng giữ `h1`, `h2`. | Điều kiện vòng lặp: `while i < len(left) or j < len(right)`. | 3 nhánh: `j == len(right) or (i < len(left) and left[i][0] < right[j][0])` → lấy L; nhánh R tương tự; còn lại là trùng x. |
| 5 | Skyline là hàm bậc thang: diện tích = tổng các hình chữ nhật. | `zip(sky, sky[1:])` cho từng cặp liên tiếp. | `sum((x2 - x1) * y1 for (x1, y1), (x2, y2) in zip(sky, sky[1:]))` |
| 6 | Chiều cao tại q = y của key point cuối cùng có x ≤ q. | `bisect_right(key_xs, q)` = số key point có x ≤ q. | `k = bisect.bisect_right(key_xs, q) - 1`; `out.append(sky[k][1] if k >= 0 else 0)` |

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
(-15, 7)
15
2 12
```

- Đỉnh heap là tuple nhỏ nhất: `-15` nhỏ nhất → `(-15, 7)`; đổi dấu → cao nhất là `15`.
- Tại x = 9: đỉnh `(-15, 7)` có `7 <= 9` → pop. Đỉnh mới là `(-12, 12)`, `12 <= 9` là False → **dừng**.
- `(-10, 9)` (tòa đã hết hạn tại 9) **vẫn nằm trong heap** vì nó không ở đỉnh — đó chính là xóa lười. Nên `len(heap) = 2`, không phải 1, và chiều cao đọc ra vẫn đúng là `12`.

### Đáp án Bài 2

Gọi A = `[1,5,3]`, B = `[2,4,6]`, C = `[3,7,4]`. Tọa độ quan trọng: `1, 2, 3, 4, 5, 7`; `h` lần lượt `3, 6, 6, 4, 4, 0`. Skyline: `[[1,3],[2,6],[4,4],[7,0]]`.

| x | Push | Pop | Heap sau bước | cur | Hành động |
|---:|---|---|---|---:|---|
| 1 | A | — | A3 | 3 | ghi `[1,3]` |
| 2 | B | — | B6, A3 | 6 | ghi `[2,6]` |
| 3 | C | — | B6, C4, A3 | 6 | bỏ qua |
| 4 | — | B | C4, A3 | 4 | ghi `[4,4]` |
| 5 | — | — | C4, A3(hết) | 4 | bỏ qua |
| 7 | — | C, A | (rỗng) | 0 | ghi `[7,0]` |

Tại x = 5, A đã hết hạn nhưng nằm dưới C nên vẫn ở trong heap ("rác"); nó bị vứt ở x = 7, ngay sau khi C bị vứt. Cell kế tiếp kiểm chứng Bài 1, Bài 2 bằng code (kèm hình vẽ skyline của Bài 2); cell sau nữa là đáp án Bài 3–6 có assert.

In [37]:
# Kiểm chứng Bài 1
heap = []
for h, r in [(10, 9), (15, 7), (12, 12)]:
    heapq.heappush(heap, (-h, r))
print(heap[0])
print(-heap[0][0])
x = 9
while heap and heap[0][1] <= x:
    heapq.heappop(heap)
print(len(heap), -heap[0][0])
print("heap còn:", sorted(heap))
print()

# Kiểm chứng Bài 2
b2 = [[1, 5, 3], [2, 4, 6], [3, 7, 4]]
ans2 = trace_skyline_heap(b2)
assert ans2 == skyline_bruteforce(b2) == [[1, 3], [2, 6], [4, 4], [7, 0]]
draw_skyline(b2, ans2)

(-15, 7)
15
2 12
heap còn: [(-12, 12), (-10, 9)]

x= 1 | push A   | pop -   | heap: A3                | cur= 3 prev= 0 -> ghi [1, 3]
x= 2 | push B   | pop -   | heap: B6, A3            | cur= 6 prev= 3 -> ghi [2, 6]
x= 3 | push C   | pop -   | heap: B6, C4, A3        | cur= 6 prev= 6 -> bỏ qua
x= 4 | push -   | pop B   | heap: C4, A3            | cur= 4 prev= 6 -> ghi [4, 4]
x= 5 | push -   | pop -   | heap: C4, A3(hết)       | cur= 4 prev= 4 -> bỏ qua
x= 7 | push -   | pop C,A | heap: (rỗng)            | cur= 0 prev= 4 -> ghi [7, 0]
  6 |  *#    
  5 |  ##    
  4 |  ##*## 
  3 | *##### 
  2 | ###### 
  1 | ###### 
    +-------*
     0       
     01234567


In [38]:
def answer_skyline_heap_exercise(buildings):
    xs = sorted({x for l, r, _ in buildings for x in (l, r)})
    heap = []
    i, n = 0, len(buildings)
    res, prev = [], 0
    for x in xs:
        while i < n and buildings[i][0] <= x:
            l, r, h = buildings[i]
            heapq.heappush(heap, (-h, r))
            i += 1
        while heap and heap[0][1] <= x:
            heapq.heappop(heap)
        cur = -heap[0][0] if heap else 0
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


def answer_merge_skylines_exercise(left, right):
    i = j = 0
    h1 = h2 = 0
    res, prev = [], 0
    while i < len(left) or j < len(right):
        if j == len(right) or (i < len(left) and left[i][0] < right[j][0]):
            x, h1 = left[i]
            i += 1
        elif i == len(left) or right[j][0] < left[i][0]:
            x, h2 = right[j]
            j += 1
        else:
            x, h1 = left[i]
            h2 = right[j][1]
            i += 1
            j += 1
        cur = max(h1, h2)
        if cur != prev:
            res.append([x, cur])
            prev = cur
    return res


def answer_skyline_area(buildings):
    sky = Solution().getSkyline(buildings)
    return sum((x2 - x1) * y1 for (x1, y1), (x2, y2) in zip(sky, sky[1:]))


def answer_heights_at(buildings, queries):
    sky = Solution().getSkyline(buildings)
    key_xs = [x for x, _ in sky]
    out = []
    for q in queries:
        k = bisect.bisect_right(key_xs, q) - 1     # key point cuối cùng có x <= q
        out.append(sky[k][1] if k >= 0 else 0)
    return out


# Bài 3
assert answer_skyline_heap_exercise(ex1) == out1
assert answer_skyline_heap_exercise(ex2) == out2
assert answer_skyline_heap_exercise([[1, 10, 5], [2, 3, 8]]) == [[1, 5], [2, 8], [3, 5], [10, 0]]
assert answer_skyline_heap_exercise([[1, 2, 1], [3, 4, 1]]) == [[1, 1], [2, 0], [3, 1], [4, 0]]
# Bài 4
assert answer_merge_skylines_exercise([[2, 10], [3, 15], [7, 10], [9, 0]], [[5, 12], [12, 0]]) == [[2, 10], [3, 15], [7, 12], [12, 0]]
assert answer_merge_skylines_exercise([[0, 3], [2, 0]], [[2, 3], [5, 0]]) == [[0, 3], [5, 0]]
assert answer_merge_skylines_exercise([[1, 1], [2, 0]], [[3, 1], [4, 0]]) == [[1, 1], [2, 0], [3, 1], [4, 0]]
assert answer_merge_skylines_exercise([], [[1, 2], [3, 0]]) == [[1, 2], [3, 0]]
# Bài 5
assert answer_skyline_area(ex1) == 212
assert answer_skyline_area(ex2) == 15
assert answer_skyline_area([[1, 10, 5], [2, 3, 8]]) == 48
# Bài 6
assert answer_heights_at(ex1, [0, 2, 3, 6, 7, 11, 12, 14, 19, 23, 24, 100]) == [0, 10, 15, 15, 12, 12, 0, 0, 10, 8, 0, 0]
assert answer_heights_at(ex2, [2, 0, 5, 4]) == [3, 3, 0, 3]

# Đối chiếu thêm với input ngẫu nhiên (trọng tài: tô từng ô)
random.seed(2024)
for _ in range(500):
    b = random_buildings()
    assert answer_skyline_heap_exercise(b) == skyline_grid(b)
    mid = len(b) // 2
    if mid > 0:
        merged = answer_merge_skylines_exercise(skyline_divide_conquer(b[:mid]), skyline_divide_conquer(b[mid:]))
        assert merged == skyline_grid(b)
    lo, hi = min(l for l, _, _ in b), max(r for _, r, _ in b)
    assert answer_skyline_area(b) == sum(height_at(b, x) for x in range(lo, hi))
    qs = [random.randint(-2, 17) for _ in range(10)]
    assert answer_heights_at(b, qs) == [height_at(b, q) for q in qs]

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 18. Template cuối cùng nên nhớ

**Cách chính — sweep line + max-heap + xóa lười** (`O(n log n)` time, `O(n)` space):

```python
import heapq

class Solution:
    def getSkyline(self, buildings: List[List[int]]) -> List[List[int]]:
        xs = sorted({x for l, r, _ in buildings for x in (l, r)})
        heap, res, prev = [], [], 0          # heap chứa (-height, right)
        i, n = 0, len(buildings)
        for x in xs:
            while i < n and buildings[i][0] <= x:      # 1. cho tòa mới vào
                l, r, h = buildings[i]
                heapq.heappush(heap, (-h, r))
                i += 1
            while heap and heap[0][1] <= x:             # 2. vứt đỉnh hết hạn
                heapq.heappop(heap)
            cur = -heap[0][0] if heap else 0            # 3. đọc đỉnh
            if cur != prev:                             # 4. đổi thì ghi
                res.append([x, cur])
                prev = cur
        return res
```

**Câu thần chú**: *"Dừng ở mọi mép — cho tòa mới vào — vứt đỉnh hết hạn — đọc đỉnh — đổi thì ghi."*

**Bản sự kiện gọn** (nhớ mã hóa VÀO = `(l, -h, r)`, RA = `(r, 0, 0)`, hai lính canh):

```python
events = sorted([(l, -h, r) for l, r, h in buildings] + [(r, 0, 0) for _, r, _ in buildings])
res, heap = [[0, 0]], [(0, float("inf"))]
for x, neg_h, r in events:
    while heap[0][1] <= x:
        heapq.heappop(heap)
    if neg_h:
        heapq.heappush(heap, (neg_h, r))
    if res[-1][1] != -heap[0][0]:
        res.append([x, -heap[0][0]])
return res[1:]
```

**Chia để trị**: *"1 tòa → `[[l,h],[r,0]]`; chia đôi; merge hai skyline bằng hai con trỏ — trùng x thì cập nhật cả hai — ghi khi max đổi."*

## 19. Checklist tự kiểm tra trước khi submit

- Mình có dùng nửa khoảng `[l, r)` — dọn đỉnh với `heap[0][1] <= x` (không phải `<`) chưa?
- Mình có gom **mọi** thay đổi tại cùng một x trước khi đọc chiều cao (hoặc dùng đúng mã hóa sự kiện `(l, -h)` / `(r, h)`) chưa?
- Đọc max-heap có đổi dấu `-heap[0][0]` chưa?
- Có xử lý heap rỗng (mặt đất = 0) hoặc đặt lính canh chưa?
- Chỉ ghi key point khi chiều cao **đổi** so với `prev` chưa?
- Output có kết thúc bằng `[max right, 0]` và có các khoảng mặt đất giữa hai cụm tòa (như `[12, 0]`) chưa?
- Đã test: Example 1, 2; tòa cùng `left`; tòa lồng nhau; tòa chạm nhau cùng/khác chiều cao; khoảng trống; cùng `right`; tọa độ cực lớn `2^31 - 1` chưa?
- Bản gom theo x dựa vào `buildings` đã sắp theo `left` — nếu dùng cho input không bảo đảm điều đó, đã `sorted` trước chưa?